# Práctica · Primera carga distribuida del acueducto con Spark

**IFPN0025 · Big Data e Ingeniería de Datos** · Universidad Ean · Módulo 2 · Sesión 15

**Qué hace este cuaderno.** Carga la capa refinada del lago del acueducto con Spark en modo local, la agrega por día y sector, reconcilia el resultado **sin filtro de calidad** contra una referencia independiente y deja una bitácora con lo que la interfaz de Spark muestra: particiones, etapas, tareas y bytes de mezcla. Con esas cifras decide cuántas particiones usar, qué operación de agregación elegir y si conviene cachear.

| Nivel | Nombre | Dónde | Tiempo | Qué produce |
|---|---|---|---|---|
| 1 | Guiado | En clase | 45 min | Contraste con la solución de referencia de la sesión anterior; sesión `local[4]`; carga de la refinada con el error de nanosegundos resuelto; `reduceByKey` frente a `groupByKey` en bytes; reconciliación sin filtro; caché; tabla de mediciones |
| 2 | Aplicado | Autónomo · obligatorio | 1,5 h del presupuesto de 3 h de la tarea | Ampliación del equipo; tres funciones completadas en `pipeline/carga_spark.py`; dos corridas idempotentes por volumen; barrido de particiones con hipótesis; tarea que lee más bytes; evidencia y registro de decisiones |
| 3 | Frontera | Autónomo · opcional | Fuera de las 3 h | Presupuesto de memoria y formato de entrada para la carga nocturna de la capa cruda |
| · | Reto de diseño | Autónomo · opcional | 30 min | Dimensionar el procesamiento de 7,8 GB con cálculo propio |

**Cómo usarlo.** Ejecuten las celdas en orden. El Nivel 1 está resuelto para seguirlo en clase. En el Nivel 2, el Nivel 3 y el reto encontrarán celdas con `# TODO` y `NotImplementedError` que deben completar (al terminar cada una, borren la línea `raise NotImplementedError`); después de cada ejercicio hay una celda de autoverificación con `assert` que les dice si su resultado cumple el criterio de correctitud. Las decisiones de ingeniería de cada nivel están en la guía; aquí se documentan en las celdas de texto marcadas como **Su decisión**. Las celdas de código usan la sesión `spark` del sub-paso B: si cierran la sesión o reinician el núcleo, vuelvan a ejecutar desde la configuración común.

**Entorno.** Perfil `s15` del paquete en marcha (`docker compose --profile s15 up -d`, servicio `healthy`), con este cuaderno guardado en la carpeta `practica/` y abierto desde JupyterLab (http://localhost:8888). Python 3.11, Java 11 y PySpark 3.5.3 ya vienen en la imagen. Semilla fija `SEED = 42`. La sesión usa `local[4]` y `spark.driver.memory=1g`. Su pico llega a 2,15 a 2,28 GiB (2 200 a 2 330 MiB) dentro del contenedor, que está limitado a 3 GiB. Los tiempos de reloj son orientativos: varían hasta ×3 según la carga del equipo; lo estable son los bytes y las tareas. La interfaz de Spark se abre en http://localhost:4040 mientras la sesión esté viva.

**La regla de la práctica.** Ninguna decisión de particiones, de operación de agregación o de caché se declara justificada sin una cifra leída en la interfaz de Spark (tareas, bytes de mezcla, bytes leídos por tarea, Storage). Y ningún agregado se declara correcto sin reconciliarlo, sin filtro de calidad, contra una referencia independiente.

### 0. Configuración común

La celda localiza el paquete, importa el script de la carga (`pipeline/carga_spark.py`) para reutilizar sus funciones ya resueltas y calcula el hash del `datalake/` heredado: al terminar, esa carpeta debe seguir idéntica.

In [1]:
# Configuracion comun: importaciones, semilla, rutas del paquete y ayudantes
import contextlib
import datetime as dt
import hashlib
import importlib
import inspect
import io
import json
import os
import statistics
import subprocess
import sys
import tempfile
import time
from pathlib import Path

import pandas as pd
from pyspark.sql import functions as F

SEED = 42                                          # semilla fija del proyecto (la usan los generadores de datos)
EQUIPO = os.environ.get("EQUIPO", "equipo-1")      # cambien el valor por el identificador de su equipo (equipo-1 a equipo-6)
DRIVER_MEMORY = "1g"                               # memoria de la JVM del controlador (ver el experimento opcional del sub-paso B)

# Raiz del paquete: la carpeta que contiene datalake/ y pipeline/. Este cuaderno vive en practica/.
for candidata in (Path.cwd(), Path.cwd().parent):
    if (candidata / "datalake").is_dir() and (candidata / "pipeline").is_dir():
        RAIZ = candidata
        break
else:
    raise FileNotFoundError("No encuentro datalake/ y pipeline/: abran este cuaderno desde la carpeta practica/ del paquete")
sys.path.insert(0, str(RAIZ / "pipeline"))
import carga_spark as cs    # el script de la carga: de el se reutilizan la sesion, la lectura de la interfaz, el hash y la reconciliacion

LAGO = RAIZ / "datalake"
REFINADA = LAGO / "refinada" / "acueducto"


def hash_arbol(carpeta, excluir=("consolidada_spark",)):
    """SHA-256 de un arbol de archivos: combina cada ruta relativa con el hash de su contenido, en orden.
    Sirve para comprobar que una carpeta NO cambio. Devuelve (hash, numero de archivos)."""
    h, n = hashlib.sha256(), 0
    for archivo in sorted(Path(carpeta).rglob("*")):
        relativa = archivo.relative_to(carpeta).as_posix()
        if archivo.is_file() and not relativa.startswith(excluir) and not archivo.name.endswith(".crc"):
            h.update(relativa.encode())
            h.update(hashlib.sha256(archivo.read_bytes()).hexdigest().encode())
            n += 1
    return h.hexdigest(), n


# Ayudantes que leen la interfaz de Spark por su API REST (la misma que alimenta las pestanas Jobs, Stages, Storage y Executors).
# Usan la variable global "spark": se asigna al abrir la sesion, mas abajo.
def api(ruta):
    return cs._rest(spark, ruta)


def siguiente_trabajo():
    """Identificador del proximo trabajo: sirve para contar solo lo que ocurre despues."""
    return max([j["jobId"] for j in api("jobs")], default=-1) + 1


def actividad(desde_trabajo):
    """Trabajos, etapas y tareas completados desde un trabajo dado."""
    time.sleep(0.5)    # la interfaz recibe los eventos de forma asincrona: se espera medio segundo
    trabajos = [j for j in api("jobs") if j["jobId"] >= desde_trabajo]
    return {"trabajos": len(trabajos),
            "etapas": sum(j["numCompletedStages"] for j in trabajos),
            "tareas": sum(j["numCompletedTasks"] for j in trabajos)}


print("Raiz del paquete:", RAIZ)
print("Semilla:", SEED, "· equipo:", EQUIPO, "· memoria del controlador:", DRIVER_MEMORY)
datalake_antes, archivos_antes = hash_arbol(LAGO)
print(f"datalake/ heredado: {archivos_antes} archivos · SHA-256 {datalake_antes[:16]}...")

Raiz del paquete: /proyecto
Semilla: 42 · equipo: equipo-1 · memoria del controlador: 1g
datalake/ heredado: 360 archivos · SHA-256 0a986ffec04ad666...


## Nivel 1 · Guiado · 45 minutos

Se resuelve sobre la **base común** (200 medidores, 120 días, 576 000 lecturas): la capa refinada en Parquet es la ruta principal; la cruda en CSV solo aparece como contraste. Las celdas de este nivel ocupan unos 3 minutos de ejecución; el resto del tiempo es lectura de la interfaz y decisión. Si no alcanzan a escribir el contraste del sub-paso A o las celdas **Su decisión**, pueden diferirlos al comienzo del Nivel 2.

### Sub-paso A · Contraste con la solución de referencia de la captura de cambios (4 minutos)

La celda siguiente imprime lo que el código de referencia (`pipeline/cdc_s14.py`) declara de sí mismo. Con esa lectura y con su memo de la sesión anterior, completen en su repositorio la tabla de contraste de la guía: una fila por decisión, con lo que mantienen, lo que cambiarían y por qué. Una decisión distinta de la de referencia no es incorrecta si está bien justificada.

In [2]:
# Sub-paso A: lean lo que la solucion de referencia de la captura de cambios dice de si misma
import ast
documentacion = ast.get_docstring(ast.parse((RAIZ / "pipeline" / "cdc_s14.py").read_text(encoding="utf-8")))
print(documentacion.split("Funciones")[0])

Prototipo de referencia de captura de cambios (CDC) por marca temporal.

Que hace
    Captura los cambios de maestro_suscriptores con el metodo mas simple: sondear la
    tabla con la marca temporal de ultima modificacion.
      1. Lee el checkpoint del equipo en estado_cdc_suscriptores. Si aun no hay ninguno
         usa un centinela antiguo (2000-01-01), nunca NULL: la primera corrida ve toda
         la tabla sin un caso especial.
      2. Consulta WHERE updated_at > checkpoint.
      3. Clasifica cada fila: created_at igual a updated_at es una ALTA (la fila nunca
         existio antes del checkpoint); created_at menor es una MODIFICACION.
      4. Agrega un registro por cambio a stg_cambios_suscriptor (solo inserciones) y
         avanza el checkpoint al mayor updated_at leido, en UNA sola transaccion: si el
         proceso cae en medio no queda un staging escrito con el checkpoint sin avanzar
         ni al reves.
      5. Devuelve el conteo por tipo de cambio. bajas_capturadas 

**Su decisión · Sub-paso A (equipo-1).** Contraste de nuestra T14 con la referencia `pipeline/cdc_s14.py`:

| Decisión | Referencia | Nuestra T14 | Qué mantenemos y qué cambiaríamos |
|---|---|---|---|
| Nomenclatura del checkpoint | `estado_cdc_suscriptores`, con una fila por equipo | Igual | **Coincide.** La mantenemos porque los nombres predecibles los consumen las etapas siguientes. |
| Valor guardado como checkpoint | El mayor `updated_at` leído, con un centinela de 2000-01-01 | Igual, y además proponemos `GREATEST()` para que el checkpoint nunca retroceda | **Coincide** en el valor. Agregaríamos `GREATEST`, porque nuestra repetición con el checkpoint restablecido duplicó 21 filas. |
| Frontera de la transacción | Cambios y checkpoint en una sola transacción | Igual | **Coincide.** |
| Método de captura | Consulta por marca temporal | Consulta horaria más reconciliación de llaves cada 6 h | Mantenemos la estrategia híbrida: con nuestras 11 bajas (34 % de los cambios), la consulta sola no alcanza. |
| Bajas duras | Se miden con una consulta por `corrida_id` y la referencia nombra tres técnicas | Medimos 0 de 11 y elegimos la reconciliación (420 llaves al día) | Coincide en medir y no suponer. Nuestra elección está justificada con las cifras del perfil. |


### Sub-paso B · Levantar el perfil `s15` y abrir la interfaz de Spark (4 minutos)

Desde la terminal del computador: `docker compose --profile s12 down`, luego `docker compose --profile s15 up -d`. En la sesión `local[4]` todo vive en una sola JVM: el único "ejecutor" es el propio controlador. Por eso `spark.executor.memory` no tiene efecto y la memoria de almacenamiento sale de `spark.driver.memory`: $M=(H-300\,\text{MiB})\cdot 0{,}6$ con $H=1024$ MiB.

In [3]:
# B.1 Comprobacion del entorno: que versiones corren dentro del contenedor
import platform
import pyspark

print("Python", platform.python_version(), "· PySpark", pyspark.__version__)
print(subprocess.run(["java", "-version"], capture_output=True, text=True).stderr.splitlines()[0])
print("Nucleos visibles para el contenedor:", os.cpu_count())

Python 3.11.17 · PySpark 3.5.3
openjdk version "11.0.32.1" 2026-08-18
Nucleos visibles para el contenedor: 6


In [4]:
# B.2 La sesion de Spark: local[4] = una sola JVM con 4 ranuras de tareas; crear_sesion ya fija UTC y la memoria
spark = cs.crear_sesion("local[4]", DRIVER_MEMORY, "true")
sc = spark.sparkContext
print("Spark", spark.version, "· master:", sc.master, "· paralelismo por omision:", sc.defaultParallelism)
print("Interfaz de Spark: http://localhost:" + sc.uiWebUrl.rsplit(":", 1)[-1])
print("Memoria del controlador:", sc.getConf().get("spark.driver.memory"), "· zona horaria de la sesion:", spark.conf.get("spark.sql.session.timeZone"))

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/12 21:49:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 3.5.3 · master: local[4] · paralelismo por omision: 4
Interfaz de Spark: http://localhost:4040


Memoria del controlador: 1g · zona horaria de la sesion: UTC


In [5]:
# B.3 Pestana Executors (por su API): en modo local hay UN solo ejecutor, llamado "driver"
ejecutores = api("allexecutors")
for e in ejecutores:
    print(f"ejecutor {e['id']}: {e['totalCores']} ranuras · memoria de almacenamiento maxima {e['maxMemory'] / 2**20:.1f} MiB")
maxima_mib = round(ejecutores[0]["maxMemory"] / 2**20, 1)
print("Formula M = (H - 300 MiB) * 0,6 con H = 1024 MiB:", round((1024 - 300) * 0.6, 1), "MiB")

ejecutor driver: 4 ranuras · memoria de almacenamiento maxima 434.4 MiB
Formula M = (H - 300 MiB) * 0,6 con H = 1024 MiB: 434.4 MiB


In [6]:
# Autoverificacion del sub-paso B
assert len(ejecutores) == 1 and ejecutores[0]["id"] == "driver", "en modo local hay un unico ejecutor: el controlador"
assert ejecutores[0]["totalCores"] == 4, "local[4] ofrece 4 ranuras de tareas"
if DRIVER_MEMORY == "1g":
    assert abs(maxima_mib - 434.4) < 0.2, "con 1g, la memoria de almacenamiento es (1024 - 300) * 0,6 = 434,4 MiB"
print("Autoverificacion del sub-paso B superada: un ejecutor, 4 ranuras y", maxima_mib, "MiB de almacenamiento.")

Autoverificacion del sub-paso B superada: un ejecutor, 4 ranuras y 434.4 MiB de almacenamiento.


**Experimento opcional (2 minutos).** Cambien `DRIVER_MEMORY = "2g"` en la celda de configuración, **reinicien el núcleo** (Kernel > Restart) y vuelvan a ejecutar hasta la pestaña Executors: la memoria de almacenamiento pasa de 434,4 a 1 048,8 MiB, porque $(2048-300)\cdot 0{,}6=1\,048{,}8$. Cambiarla sin reiniciar el núcleo no surte efecto: la JVM ya arrancó con 1g y `getOrCreate()` devuelve la sesión existente. Devuelvan el valor a `"1g"` antes de seguir: con `2g` el pico de memoria del contenedor se acerca al límite.

**Su decisión · Sub-paso B.** Usamos `local[4]` y no `local[*]` porque el contenedor tiene un tope de 3 GiB. En modo local, cada ranura de tareas que ejecuta código Python abre su propio proceso Python, de unos 90 MiB, aparte de la JVM. La pestaña Executors muestra **un solo ejecutor (`driver`), con 4 ranuras y 434,4 MiB de memoria de almacenamiento** ((1024 − 300) × 0,6). `nproc` dentro del contenedor reporta 6 núcleos: `local[*]` abriría 6 ranuras, 6 procesos de Python y más presión sobre esos 434 MiB, sin ganar nada con 4 particiones de entrada, porque una quinta y una sexta ranura quedarían ociosas.

### Sub-paso C · Cargar la refinada: marca temporal en nanosegundos, particiones y contraste con la cruda (12 minutos)

La refinada guarda `timestamp` en **nanosegundos**, y Spark 3.5 no lo lee como marca temporal. Primero se provoca el error y se lee el mensaje; luego se activa `spark.sql.legacy.parquet.nanosAsLong`, que entrega la columna como entero largo, y se convierte con `timestamp_seconds(timestamp DIV 1000000000)`. La conversión se verifica a mano con una fila, se cuentan las particiones de Spark (que no son las carpetas `dia=DD`) y se fija la zona horaria en UTC para que la fecha derivada coincida con la carpeta.

In [7]:
# C.1 Provocar el error: la refinada guarda la marca temporal en NANOSEGUNDOS y Spark 3.5 no la lee como marca temporal.
# crear_sesion ya activaba la opcion nanosAsLong: la apagamos para ver que pasa sin ella.
spark.conf.set("spark.sql.legacy.parquet.nanosAsLong", "false")
trabajo_previo = siguiente_trabajo()
try:
    spark.read.parquet(str(REFINADA))
    mensaje_nanos = ""
    print("La lectura no fallo: revisen que nanosAsLong quedo en false")
except Exception as error:
    lineas_error = [l[l.index("Illegal Parquet type"):].strip() for l in str(error).splitlines() if "Illegal Parquet type" in l]
    mensaje_nanos = lineas_error[0] if lineas_error else str(error)[:400]
    print("Excepcion de Python:", type(error).__name__)
    print("Mensaje de Spark:", mensaje_nanos)
fallidos = [(j["jobId"], j["status"]) for j in api("jobs") if j["jobId"] >= trabajo_previo]
print("Trabajos lanzados por la lectura (el error ocurre al LEER, no al contar):", fallidos)

Excepcion de Python: Py4JJavaError
Mensaje de Spark: Illegal Parquet type: INT64 (TIMESTAMP(NANOS,false)).
Trabajos lanzados por la lectura (el error ocurre al LEER, no al contar): [(0, 'FAILED')]


26/11/12 21:49:29 ERROR Executor: Exception in task 0.0 in stage 0.0 (TID 0)
org.apache.spark.sql.AnalysisException: Illegal Parquet type: INT64 (TIMESTAMP(NANOS,false)).
	at org.apache.spark.sql.errors.QueryCompilationErrors$.illegalParquetTypeError(QueryCompilationErrors.scala:1825)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetToSparkSchemaConverter.illegalType$1(ParquetSchemaConverter.scala:199)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetToSparkSchemaConverter.$anonfun$convertPrimitiveField$2(ParquetSchemaConverter.scala:276)
	at scala.Option.getOrElse(Option.scala:189)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetToSparkSchemaConverter.convertPrimitiveField(ParquetSchemaConverter.scala:217)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetToSparkSchemaConverter.convertField(ParquetSchemaConverter.scala:180)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetToSparkSchemaConverter.$anonfun$convertInternal$3

In [8]:
# C.2 Con la opcion activada, la marca temporal llega como entero largo: nanosegundos desde la epoca
spark.conf.set("spark.sql.legacy.parquet.nanosAsLong", "true")
trabajo0 = siguiente_trabajo()
df = spark.read.parquet(str(REFINADA))      # lee la carpeta COMPLETA: Spark descubre anio, mes y dia desde la ruta
df.printSchema()
actividad_lectura = actividad(trabajo0)
print("Lo que lanzo la lectura, antes de cualquier accion:", actividad_lectura)

root
 |-- meter_id: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- sector: string (nullable = true)
 |-- consumo_m3: double (nullable = true)
 |-- presion_bar: double (nullable = true)
 |-- quality_flag: string (nullable = true)
 |-- anio: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- dia: integer (nullable = true)



Lo que lanzo la lectura, antes de cualquier accion: {'trabajos': 1, 'etapas': 1, 'tareas': 1}


In [9]:
# C.3 Conversion a marca temporal: segundos = nanosegundos DIV 10^9
df_ts = df.withColumn("ts", F.expr("timestamp_seconds(`timestamp` DIV 1000000000)"))
tipos = dict(df_ts.dtypes)
print("Tipos:", {c: tipos[c] for c in ("timestamp", "ts", "anio", "mes", "dia")})
df_ts.select("meter_id", "timestamp", "ts", "anio", "mes", "dia").limit(3).show(truncate=False)

# Verificacion a mano de UNA fila: el mismo calculo en Python puro, en UTC
fila = df_ts.select("timestamp", F.date_format("ts", "yyyy-MM-dd HH:mm:ss").alias("ts_texto")).limit(1).collect()[0]
a_mano = dt.datetime.fromtimestamp(fila["timestamp"] // 10**9, dt.timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
print("Spark:", fila["ts_texto"], "· Python:", a_mano)

Tipos: {'timestamp': 'bigint', 'ts': 'timestamp', 'anio': 'int', 'mes': 'int', 'dia': 'int'}


+--------+-------------------+-------------------+----+---+---+
|meter_id|timestamp          |ts                 |anio|mes|dia|
+--------+-------------------+-------------------+----+---+---+
|MED0001 |1704931200000000000|2024-01-11 00:00:00|2024|1  |11 |
|MED0001 |1704934800000000000|2024-01-11 01:00:00|2024|1  |11 |
|MED0001 |1704938400000000000|2024-01-11 02:00:00|2024|1  |11 |
+--------+-------------------+-------------------+----+---+---+

Spark: 2024-01-11 00:00:00 · Python: 2024-01-11 00:00:00


In [10]:
# C.4 Particiones de Spark (no son las carpetas dia=DD) y evaluacion perezosa
rdd_filas = df_ts.rdd
particiones_parquet = rdd_filas.getNumPartitions()
carpetas_dia = len(list(REFINADA.glob("anio=*/mes=*/dia=*")))
print("Particiones de Spark:", particiones_parquet, "· carpetas dia=DD del lago:", carpetas_dia)

trabajo1 = siguiente_trabajo()
transformado = rdd_filas.map(lambda f: (f.sector, f.consumo_m3)).filter(lambda kv: kv[1] >= 0)   # dos transformaciones
actividad_transformaciones = actividad(trabajo1)
print("Tras definir dos transformaciones:", actividad_transformaciones)

trabajo2 = siguiente_trabajo()
lecturas_totales = transformado.count()                                                          # una accion
actividad_count = actividad(trabajo2)
print("Tras count():", actividad_count, "· lecturas:", lecturas_totales)

Particiones de Spark: 4 · carpetas dia=DD del lago: 120


Tras definir dos transformaciones: {'trabajos': 0, 'etapas': 0, 'tareas': 0}


Tras count(): {'trabajos': 1, 'etapas': 1, 'tareas': 4} · lecturas: 576000


In [11]:
# C.5 Zona horaria: la fecha derivada del instante debe coincidir con la carpeta dia=DD de donde salio la fila
def filas_con_fecha_distinta(zona):
    spark.conf.set("spark.sql.session.timeZone", zona)
    return df_ts.filter("CAST(to_date(ts) AS STRING) != concat(anio, '-', lpad(mes, 2, '0'), '-', lpad(dia, 2, '0'))").count()

distintas_utc = filas_con_fecha_distinta("UTC")
distintas_bogota = filas_con_fecha_distinta("America/Bogota")
spark.conf.set("spark.sql.session.timeZone", "UTC")      # se restablece la zona de la sesion
print("Filas cuya fecha difiere de la carpeta con zona UTC:", distintas_utc)
print("Filas cuya fecha difiere de la carpeta con zona America/Bogota (UTC-5):", distintas_bogota)

Filas cuya fecha difiere de la carpeta con zona UTC: 0
Filas cuya fecha difiere de la carpeta con zona America/Bogota (UTC-5): 120000


In [12]:
# C.6 Contraste (2 min): la capa cruda en CSV, leida como texto con textFile
trabajo3 = siguiente_trabajo()
cruda = cs.leer_cruda_csv(spark, LAGO / "cruda")
lecturas_cruda = cruda.count()
actividad_cruda = actividad(trabajo3)
particiones_cruda = cruda.getNumPartitions()
print("Particiones de la cruda (textFile):", particiones_cruda, "· Parquet:", particiones_parquet)
print("Lecturas:", lecturas_cruda, "· lo que lanzo count():", actividad_cruda)

Particiones de la cruda (textFile): 120 · Parquet: 4
Lecturas: 576000 · lo que lanzo count(): {'trabajos': 1, 'etapas': 1, 'tareas': 120}


In [13]:
# Autoverificacion del sub-paso C: el comportamiento manda; las cifras medidas se informan
assert "Illegal Parquet type" in mensaje_nanos, "sin nanosAsLong, Spark rechaza el tipo de la marca temporal al LEER"
assert tipos["timestamp"] == "bigint" and tipos["ts"] == "timestamp", "timestamp llega como entero largo y ts como marca temporal"
assert {"anio", "mes", "dia"} <= set(df.columns), "al leer la carpeta completa aparecen anio, mes y dia"
assert fila["ts_texto"] == a_mano, "la conversion de Spark coincide con el calculo a mano"
assert lecturas_totales == 576_000 and lecturas_cruda == 576_000
assert actividad_transformaciones["trabajos"] == 0, "las transformaciones no lanzan trabajos"
assert actividad_count["trabajos"] == 1 and actividad_count["tareas"] == particiones_parquet
assert actividad_cruda["tareas"] == particiones_cruda == 120, "con textFile hay una particion por archivo: 120"
assert distintas_utc == 0 and distintas_bogota > 0, "con UTC la fecha derivada coincide con la carpeta; con UTC-5 no"
print("Autoverificacion del sub-paso C superada: el error de nanosegundos se provoco, se resolvio y se leyo la actividad de Spark.")

Autoverificacion del sub-paso C superada: el error de nanosegundos se provoco, se resolvio y se leyo la actividad de Spark.


**Su decisión · Sub-paso C.** Fijamos `spark.sql.session.timeZone=UTC` porque la fecha se deriva de un instante. Medido: con UTC, **0** filas tienen una fecha distinta de su carpeta `dia=DD`; con `America/Bogota` son **120 000**, es decir, las 5 primeras horas de cada día pasan al día anterior. Sin fijar la zona, el agregado dependería del computador donde corre. Las **4 particiones de entrada no las elegimos directamente**: Spark las calcula a partir del tamaño de los archivos (`spark.sql.files.maxPartitionBytes`, 128 MiB), del costo de abrir cada archivo y del paralelismo por omisión (4 ranuras). Por eso 120 Parquet pequeños se empaquetan en 4 particiones, mientras que `textFile` sobre la cruda da 120, una por archivo.

### Sub-paso D · `reduceByKey` frente a `groupByKey`, reconciliación y `groupBy` de DataFrame (13 minutos)

Los dos métodos producen el mismo resultado; lo que cambia es **cuánto viaja entre etapas**. La interfaz cuenta registros y bytes de mezcla: los registros (PySpark los agrupa en lotes serializados) no distinguen los métodos; los bytes sí. Después se reconcilia el agregado **sin filtro de calidad** contra `consolidada/` y se contrasta con un agregado filtrado, que tiene sus propios totales. Cierra una demostración del `groupBy` de DataFrame con la ejecución adaptativa activa y desactivada.

In [14]:
# D.1 Pares ((fecha, sector), consumo): la clave es compuesta y hay 120 dias x 4 sectores = 480 claves
lecturas = df.selectExpr("CAST(to_date(timestamp_seconds(`timestamp` DIV 1000000000)) AS STRING) AS fecha", "sector", "consumo_m3")
pares = lecturas.rdd.map(lambda fila: ((fila.fecha, fila.sector), fila.consumo_m3))
print("Particiones de entrada:", pares.getNumPartitions(), "· primer par:", pares.first())

Particiones de entrada: 4 · primer par: (('2024-01-11', 'S-NORTE'), 0.233)


In [15]:
# D.2 reduceByKey frente a groupByKey: los dos suman y cuentan, pero mezclan cosas muy distintas
def medir_agregacion(resultado, nombre):
    """Ejecuta collect() y lee de la interfaz que ocurrio durante esa accion."""
    etapa0 = cs.siguiente_etapa(spark)
    t0 = time.time()
    filas = resultado.collect()
    tiempo = round(time.time() - t0, 2)
    time.sleep(0.5)
    etapas = [e for e in api("stages") if e["stageId"] >= etapa0 and e["status"] == "COMPLETE"]
    medicion = {"operacion": nombre, "tiempo_s": tiempo, **cs.metricas_interfaz(spark, etapa0),
                "registros_escritos": sum(e["shuffleWriteRecords"] for e in etapas)}
    return filas, medicion


# reduceByKey: cada particion combina (suma, cuenta) por clave ANTES de mezclar
combinado = pares.map(lambda kv: (kv[0], (kv[1], 1))).reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
filas_reduce, med_reduce = medir_agregacion(combinado.map(lambda kv: (kv[0][0], kv[0][1], kv[1][0], kv[1][1])), "reduceByKey")

# groupByKey: mezcla TODOS los valores y suma despues
agrupado = pares.groupByKey().map(lambda kv: (kv[0][0], kv[0][1], sum(kv[1]), len(kv[1])))
filas_group, med_group = medir_agregacion(agrupado, "groupByKey")

comparacion = pd.DataFrame([med_reduce, med_group]).set_index("operacion")
print(comparacion.to_string())
razon_bytes = round(med_group["mezcla_escritura_bytes"] / med_reduce["mezcla_escritura_bytes"], 1)
print(f"groupByKey escribe {razon_bytes} veces mas bytes de mezcla que reduceByKey; los registros son {med_reduce['registros_escritos']} y {med_group['registros_escritos']}")

             tiempo_s  etapas  tareas  mezcla_escritura_bytes  mezcla_lectura_bytes  registros_escritos
operacion                                                                                              
reduceByKey      2.27       2       8                    9803                  9803                  48
groupByKey       1.67       2       8                 1867585               1867585                  48
groupByKey escribe 190.5 veces mas bytes de mezcla que reduceByKey; los registros son 48 y 48


In [16]:
# D.3 Reconciliacion SIN filtro de calidad contra la capa consolidada del lago
referencia = cs.referencia_base(LAGO)                  # 480 filas: fecha, sector, consumo total, lecturas
ok_reduce, detalle_reduce = cs.reconciliar(filas_reduce, referencia)
ok_group, detalle_group = cs.reconciliar(filas_group, referencia)
lecturas_por_grupo = sorted({f[3] for f in filas_reduce})
total_lecturas = sum(f[3] for f in filas_reduce)
total_consumo = round(sum(f[2] for f in filas_reduce), 3)
print("reduceByKey:", ok_reduce, "·", detalle_reduce)
print("groupByKey: ", ok_group, "·", detalle_group)
print("Filas:", len(filas_reduce), "· lecturas por sector y dia:", lecturas_por_grupo, "· total de lecturas:", total_lecturas, "· consumo total:", total_consumo, "m3")

reduceByKey: True · 480 grupos iguales a la referencia (tolerancia 1e-06)
groupByKey:  True · 480 grupos iguales a la referencia (tolerancia 1e-06)
Filas: 480 · lecturas por sector y dia: [1200] · total de lecturas: 576000 · consumo total: 92166.022 m3


In [17]:
# D.4 El filtro de calidad es una transformacion estrecha, pero cambia los totales: ese agregado NO reconcilia
solo_ok = (df.filter("quality_flag = 'OK'")
             .selectExpr("CAST(to_date(timestamp_seconds(`timestamp` DIV 1000000000)) AS STRING) AS fecha", "sector", "consumo_m3")
             .rdd.map(lambda f: ((f.fecha, f.sector), (f.consumo_m3, 1)))
             .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
             .map(lambda kv: (kv[0][0], kv[0][1], kv[1][0], kv[1][1])).collect())
lecturas_ok = sum(f[3] for f in solo_ok)
ok_filtrado, detalle_filtrado = cs.reconciliar(solo_ok, referencia)
print("Lecturas con quality_flag = 'OK':", lecturas_ok, "de", total_lecturas, f"({100 * lecturas_ok / total_lecturas:.2f} %)")
print("Reconcilia el agregado filtrado?", ok_filtrado, "·", detalle_filtrado)

Lecturas con quality_flag = 'OK': 541368 de 576000 (93.99 %)
Reconcilia el agregado filtrado? False · 480 grupos con distinto numero de lecturas y 480 con distinta suma


In [18]:
# D.5 Demostracion (3 min): el groupBy de DataFrame, con la ejecucion adaptativa (AQE) activa y desactivada
def groupby_dataframe(datos, aqe, particiones):
    """groupBy de DataFrame por dia y sector. Devuelve las tareas de la etapa de reduccion y los bytes de mezcla."""
    spark.conf.set("spark.sql.adaptive.enabled", aqe)
    spark.conf.set("spark.sql.shuffle.partitions", str(particiones))
    etapa0 = cs.siguiente_etapa(spark)
    filas = datos.groupBy("anio", "mes", "dia", "sector").agg(F.sum("consumo_m3").alias("consumo"), F.count("*").alias("lecturas")).collect()
    time.sleep(0.5)
    etapas = [e for e in api("stages") if e["stageId"] >= etapa0]
    reduccion = [e for e in etapas if e["status"] == "COMPLETE" and e["shuffleReadBytes"] > 0]
    return {"aqe": aqe, "shuffle.partitions": particiones, "filas": len(filas),
            "tareas_reduccion": reduccion[-1]["numTasks"] if reduccion else None,
            "mezcla_bytes": sum(e["shuffleWriteBytes"] for e in etapas)}


demo_aqe = pd.DataFrame([groupby_dataframe(df, "true", 200), groupby_dataframe(df, "false", 200),
                         groupby_dataframe(df, "false", 8), groupby_dataframe(df, "false", 4)])
spark.conf.set("spark.sql.adaptive.enabled", "true")          # se restablecen los valores de la sesion
spark.conf.set("spark.sql.shuffle.partitions", "200")
print(demo_aqe.to_string(index=False))

  aqe  shuffle.partitions  filas  tareas_reduccion  mezcla_bytes
 true                 200    480                 1         37888
false                 200    480               200         37888
false                   8    480                 8         13240
false                   4    480                 4         11331


In [19]:
# Autoverificacion del sub-paso D: el comportamiento manda
assert ok_reduce and ok_group, "los dos metodos deben reconciliar con la consolidada"
assert len(filas_reduce) == len(filas_group) == 480 and lecturas_por_grupo == [1200], "480 grupos de 1 200 lecturas"
assert total_lecturas == 576_000
assert med_reduce["etapas"] == med_group["etapas"] == 2, "una mezcla parte el trabajo en dos etapas"
assert med_group["mezcla_escritura_bytes"] > 50 * med_reduce["mezcla_escritura_bytes"], "groupByKey mezcla mucho mas BYTES"
assert med_reduce["registros_escritos"] == med_group["registros_escritos"], "los REGISTROS no distinguen los metodos"
assert lecturas_ok < total_lecturas and not ok_filtrado, "el agregado filtrado por calidad no reconcilia"
assert demo_aqe.iloc[0]["tareas_reduccion"] == 1, "con AQE activa la reduccion tuvo una sola tarea"
assert list(demo_aqe["tareas_reduccion"][1:]) == [200, 8, 4], "con AQE desactivada hay tantas tareas como shuffle.partitions"
assert (demo_aqe["filas"] == 480).all()
print("Autoverificacion del sub-paso D superada: reconcilia sin filtro, los bytes distinguen los metodos y los registros no.")

Autoverificacion del sub-paso D superada: reconcilia sin filtro, los bytes distinguen los metodos y los registros no.


**Su decisión · Sub-paso D.** La clave es `(fecha, sector)` porque es el grano de la referencia `consolidada/`: 120 días × 4 sectores = 480 grupos. Con otra clave no habría con qué reconciliar. `reduceByKey` es válido porque la función que combina `(suma, cuenta)` es **asociativa y conmutativa**: sumar parciales en cualquier orden y agrupación da el mismo total, hasta el redondeo de punto flotante que cubre la tolerancia de 10⁻⁶. Lo respalda la pestaña Stages: **9 803 B de mezcla con `reduceByKey` frente a 1 867 585 B con `groupByKey` (190,5 veces más)**, con el mismo resultado (480 grupos de 1 200 lecturas, reconciliados). El agregado filtrado por `quality_flag = 'OK'` (541 368 de 576 000 lecturas) **no reconcilia** en ninguno de los 480 grupos, porque sus totales son otros.

### Sub-paso E · Tres acciones con y sin caché (12 minutos)

Las tres acciones reutilizan las 541 368 lecturas de calidad `OK`. Sin caché, cada acción vuelve a leer y parsear el Parquet; con `cache()`, la primera acción guarda las particiones y las siguientes las toman de memoria. La pestaña Storage dice cuánto ocupan. El reloj varía hasta ×3 según la carga del equipo, así que una sola medición ilustra pero no concluye: en el Nivel 2 se usa la mediana de tres. Lo estable es la cifra de Storage.

In [20]:
# E.1 Las lecturas de calidad OK como RDD de tuplas, y tres acciones que la reutilizan
lecturas_ok_rdd = (df.filter("quality_flag = 'OK'")
                     .selectExpr("meter_id", "timestamp_seconds(`timestamp` DIV 1000000000) AS ts", "sector", "consumo_m3", "presion_bar")
                     .rdd.map(lambda f: (f.meter_id, f.ts, f.sector, f.consumo_m3, f.presion_bar)))


def tres_acciones(rdd):
    """Cuenta, suma el consumo y busca la presion maxima: tres acciones sobre el mismo RDD. Devuelve cada tiempo."""
    tiempos = []
    for accion in (lambda r: r.count(), lambda r: r.map(lambda x: x[3]).sum(), lambda r: r.map(lambda x: x[4]).max()):
        t0 = time.time()
        accion(rdd)
        tiempos.append(round(time.time() - t0, 2))
    return tiempos


n_ok = lecturas_ok_rdd.count()
tiempos_sin_cache = tres_acciones(lecturas_ok_rdd)
print("Lecturas OK:", n_ok)
print("SIN cache, tiempo de cada accion (s):", tiempos_sin_cache, "· total:", round(sum(tiempos_sin_cache), 1))

Lecturas OK: 541368
SIN cache, tiempo de cada accion (s): [1.99, 1.95, 1.62] · total: 5.6


In [21]:
# E.2 CON cache: la primera accion lee y guarda las particiones; las siguientes las toman de memoria
lecturas_ok_rdd.cache()
tiempos_con_cache = tres_acciones(lecturas_ok_rdd)
print("CON cache, tiempo de cada accion (s):", tiempos_con_cache, "· total:", round(sum(tiempos_con_cache), 1))

# Pestana Storage (por su API): cuanta memoria ocupa y en que nivel de almacenamiento
almacenado = api("storage/rdd")
for r in almacenado:
    print(f"{r['storageLevel']} · particiones en cache {r['numCachedPartitions']} de {r['numPartitions']} · memoria {r['memoryUsed'] / 2**20:.1f} MiB · disco {r['diskUsed'] / 2**20:.1f} MiB")
memoria_cache_mib = round(almacenado[0]["memoryUsed"] / 2**20, 1)

lecturas_ok_rdd.unpersist()                  # se libera la memoria: cachear tambien cuesta
time.sleep(0.5)
print("Despues de unpersist, RDDs en Storage:", len(api("storage/rdd")))

CON cache, tiempo de cada accion (s): [2.02, 0.18, 0.18] · total: 2.4
Memory Serialized 1x Replicated · particiones en cache 4 de 4 · memoria 9.4 MiB · disco 0.0 MiB


Despues de unpersist, RDDs en Storage: 0


In [22]:
# Autoverificacion del sub-paso E: el comportamiento manda; la tabla de tiempos es la del equipo
assert n_ok == 541_368, "las lecturas con quality_flag OK son 541 368 de 576 000"
assert tiempos_con_cache[1] < tiempos_sin_cache[1] and tiempos_con_cache[2] < tiempos_sin_cache[2], "con cache, las acciones siguientes son mas rapidas"
assert almacenado[0]["numCachedPartitions"] == almacenado[0]["numPartitions"] and almacenado[0]["diskUsed"] == 0, "toda la cache cabe en memoria, sin disco"
assert memoria_cache_mib < maxima_mib / 4, "la cache ocupa una fraccion pequena de los 434,4 MiB disponibles"
assert len(api("storage/rdd")) == 0, "unpersist libera la cache"
print("Autoverificacion del sub-paso E superada: con cache las acciones siguientes cuestan una fraccion y la memoria ocupada es pequena.")

Autoverificacion del sub-paso E superada: con cache las acciones siguientes cuestan una fraccion y la memoria ocupada es pequena.


**Su decisión · Sub-paso E.** **No** cacheamos en la carga de la tarea. La caché solo paga cuando el mismo RDD alimenta varias acciones: con tres acciones, el tiempo bajó de **5,6 s a 2,4 s** y Storage ocupó solo **9,4 MiB** de los 434,4 disponibles (en la ampliación: de 9,2 s a 4,5 s y 16,4 MiB). La carga de la tarea, en cambio, ejecuta **una sola acción** (`collect()`), así que la primera acción con caché (2,02 s frente a 1,99 s sin caché; en la ampliación, 3,89 s frente a 3,07 s) cuesta lo mismo o más que sin caché, porque además escribe la caché, y no hay acciones siguientes que la aprovechen.

### Tabla de mediciones del Nivel 1

La celda siguiente reúne lo medido en una tabla y la guarda como `mediciones_nivel1.csv` en la carpeta del paquete. Súbanla al repositorio del equipo.

In [23]:
# Tabla de mediciones del Nivel 1: lo que midieron, listo para el repositorio del equipo
mediciones = pd.DataFrame([
    ("Memoria de almacenamiento del ejecutor (MiB)", maxima_mib, "Executors"),
    ("Particiones de la refinada (Parquet)", particiones_parquet, "rdd.getNumPartitions()"),
    ("Particiones de la cruda (textFile)", particiones_cruda, "rdd.getNumPartitions()"),
    ("Trabajos, etapas y tareas del count() sobre Parquet", f"{actividad_count['trabajos']} · {actividad_count['etapas']} · {actividad_count['tareas']}", "Jobs"),
    ("Bytes de mezcla escritos con reduceByKey", med_reduce["mezcla_escritura_bytes"], "Stages"),
    ("Bytes de mezcla escritos con groupByKey", med_group["mezcla_escritura_bytes"], "Stages"),
    ("Razon groupByKey / reduceByKey (bytes)", razon_bytes, "Stages"),
    ("Registros de mezcla (reduceByKey y groupByKey)", f"{med_reduce['registros_escritos']} y {med_group['registros_escritos']}", "Stages"),
    ("Filas del agregado y lecturas por sector y dia", f"{len(filas_reduce)} · {lecturas_por_grupo[0]}", "reconciliacion"),
    ("Tareas de reduccion del groupBy de DataFrame (AQE activa)", int(demo_aqe.iloc[0]["tareas_reduccion"]), "Stages"),
    ("Tareas de reduccion con AQE desactivada y shuffle.partitions 200, 8 y 4", " · ".join(str(int(x)) for x in demo_aqe["tareas_reduccion"][1:]), "Stages"),
    ("Tiempo de las tres acciones sin cache (s)", round(sum(tiempos_sin_cache), 1), "reloj de pared"),
    ("Tiempo de las tres acciones con cache (s)", round(sum(tiempos_con_cache), 1), "reloj de pared"),
    ("Memoria de la cache en Storage (MiB)", memoria_cache_mib, "Storage"),
], columns=["medicion", "valor", "donde_se_lee"])
mediciones.to_csv(RAIZ / "mediciones_nivel1.csv", index=False, encoding="utf-8")
print(mediciones.to_string(index=False))

                                                               medicion       valor           donde_se_lee
                           Memoria de almacenamiento del ejecutor (MiB)       434.4              Executors
                                   Particiones de la refinada (Parquet)           4 rdd.getNumPartitions()
                                     Particiones de la cruda (textFile)         120 rdd.getNumPartitions()
                    Trabajos, etapas y tareas del count() sobre Parquet   1 · 1 · 4                   Jobs
                               Bytes de mezcla escritos con reduceByKey        9803                 Stages
                                Bytes de mezcla escritos con groupByKey     1867585                 Stages
                                 Razon groupByKey / reduceByKey (bytes)       190.5                 Stages
                         Registros de mezcla (reduceByKey y groupByKey)     48 y 48                 Stages
                         Filas del ag

In [24]:
# Cierre del Nivel 1: el datalake/ heredado no cambio (la carga de Spark escribe en consolidada_spark/, que no se cuenta)
datalake_despues, archivos_despues = hash_arbol(LAGO)
assert (datalake_despues, archivos_despues) == (datalake_antes, archivos_antes), "datalake/ debe seguir identico"
print(f"datalake/ identico: {archivos_despues} archivos · SHA-256 {datalake_despues[:16]}...")

datalake/ identico: 360 archivos · SHA-256 0a986ffec04ad666...


**Consolidación (individual, 2 minutos cada una).** (1) ¿Qué parte del tiempo fue cómputo y qué parte movimiento de datos? (2) Si el volumen se duplicara, ¿qué cifra de la interfaz cambiaría primero?

(1) Con `reduceByKey`, el movimiento de datos es despreciable: 9,8 KB de mezcla. Casi todo el tiempo (unos 2 s) es leer y convertir el Parquet y serializar filas hacia Python en la etapa de mapa. Con `groupByKey` la mezcla sube a 1,8 MB, pero en una sola máquina ese movimiento es memoria y disco local, y el tiempo casi no cambia (1,67 s frente a 2,27 s). (2) Lo primero que cambiaría son los **bytes de mezcla de `groupByKey`** (Shuffle Write), que crecen con el número de lecturas: en la ampliación, con 1,74 veces más lecturas, pasaron de 1,87 MB a 3,24 MB. Las tareas, las particiones de entrada y los bytes de `reduceByKey` no cambiarían, porque dependen del número de archivos y de claves, no de las filas.

## Nivel 2 · Aplicado · 1,5 horas del presupuesto de la tarea

**Presupuesto (estimado).** Estas 1,5 horas cubren de N2.1 a N2.7, con la evidencia incluida: generar la ampliación, 5 min; las tres funciones, 20 min; las cargas, 10 min; el barrido con las mediciones de apoyo (operación, caché y formato de entrada), 25 min; la tarea que lee más bytes, 10 min; el contraste con DataFrame, 5 min; las conclusiones y la evidencia, 15 min. El registro de decisiones (tres decisiones con cifras) tiene aparte los 30 minutos restantes de las 3 horas de la tarea. El contraste con la solución de referencia ya se hizo en el sub-paso A del Nivel 1; la plantilla de entrega se diligencia mientras trabajan, porque las celdas de evidencia imprimen las cifras que hay que transcribir; el README es breve. Si algo no cabe, la prioridad es el registro de decisiones, luego la bitácora y al final el README. Los tiempos de reloj varían hasta ×3 según la carga del equipo: lo estable son los bytes y las tareas.

### El encargo

Repitan el patrón del Nivel 1 sobre **la ampliación de su equipo** (300 a 600 medidores y un sector dominante que concentra entre el 30 % y el 60 % de los medidores), con el código de la carga en `pipeline/carga_spark.py`. Cambien `EQUIPO` en la celda de configuración (o definan la variable de entorno `EQUIPO`) y ejecuten en orden.

**Paso mecánico (N2.1).** La celda genera la ampliación en `datalake_equipo/` y comprueba que `datalake/` no cambió: el hash antes y después debe ser el mismo.

In [25]:
# N2.1 Generar la ampliacion del equipo y comprobar que datalake/ NO cambio
EQUIPO_CARPETA = RAIZ / "datalake_equipo"
EQUIPO_REFINADA = EQUIPO_CARPETA / "refinada" / "acueducto"

antes, archivos_antes_gen = hash_arbol(LAGO)
t0 = time.time()
salida_generador = subprocess.run([sys.executable, str(RAIZ / "practica" / "generar_ampliacion_s15.py"), "--equipo", EQUIPO],
                                  cwd=RAIZ, capture_output=True, text=True)
tiempo_generacion = round(time.time() - t0, 1)
print(salida_generador.stdout or salida_generador.stderr)
despues, archivos_despues_gen = hash_arbol(LAGO)
assert salida_generador.returncode == 0, "el generador fallo: revisen el identificador del equipo"
assert (antes, archivos_antes_gen) == (despues, archivos_despues_gen), "el generador no debe tocar datalake/"

perfil = json.loads((EQUIPO_CARPETA / "parametros_equipo.json").read_text(encoding="utf-8"))
disco_mib = round(sum(f.stat().st_size for f in EQUIPO_CARPETA.rglob("*") if f.is_file() and "consolidada_spark" not in f.parts) / 2**20, 1)
print({k: perfil[k] for k in ("equipo", "medidores", "lecturas_totales", "sector_dominante", "fraccion_dominante")})
print(f"Disco ocupado: {disco_mib} MiB · tiempo de generacion: {tiempo_generacion} s · datalake/ identico ({archivos_despues_gen} archivos)")

Equipo equipo-1: 348 medidores, 120 dias, 1,002,240 lecturas
Sector dominante S-SUR: 120 medidores (34.5%); cada uno de los otros tres: 76
Escrito en datalake_equipo (56.0 MiB, 3.1 s)

{'equipo': 'equipo-1', 'medidores': 348, 'lecturas_totales': 1002240, 'sector_dominante': 'S-SUR', 'fraccion_dominante': 0.3448}
Disco ocupado: 56.0 MiB · tiempo de generacion: 3.5 s · datalake/ identico (360 archivos)


### N2.2 Su turno: completar las tres funciones de la carga

`leer_refinada`, `agregar` y `escribir_salida` son las tres funciones que faltan. Cada celda de ejercicio va seguida de su autoverificación sobre la base, que reproduce lo que ya hicieron en el Nivel 1. Cuando las tres pasen, **llévenlas a `pipeline/carga_spark.py`** (reemplacen cada `NotImplementedError`) y ejecuten la última celda de esta sección.

**N2.2a · `leer_refinada`**

In [26]:
import datetime as dt

def leer_refinada(spark, carpeta_refinada):
    """Lee la capa refinada y devuelve un RDD de pares ((fecha, sector), consumo_m3).

    TODO (equipo):
      1. Lean la carpeta completa con spark.read.parquet. Revisen el esquema: que tipo
         tiene la marca temporal y que columnas agrego Spark a partir de la ruta?
      2. Conviertan la marca temporal (nanosegundos desde la epoca) a segundos y de ahi a
         fecha, en la zona horaria de la sesion. Escriban la conversion con una expresion
         de Spark SQL y verifiquen una fila a mano.
      3. Pasen a RDD (.rdd) y armen los pares con la fecha como texto AAAA-MM-DD.
    Una transformacion no ejecuta nada: el error de una conversion mal escrita aparece
    al ejecutar la accion, no aqui.

    Decision (equipo-1): la marca llega como entero largo de nanosegundos (por
    spark.sql.legacy.parquet.nanosAsLong=true). Se divide ENTERO entre 10^9 (div, no /,
    para no pasar por double) y timestamp_seconds la interpreta en UTC, la zona de la
    sesion; date_format da el texto AAAA-MM-DD. Verificacion a mano: 1704067200000000000 ns
    -> 1704067200 s -> 2024-01-01 00:00:00 UTC -> '2024-01-01'. Las columnas anio, mes y
    dia que Spark agrega desde la ruta (descubrimiento de particiones) no se usan: la
    fecha sale del dato, no de la carpeta. No hay filtro de calidad: se suman todas las
    lecturas, como exige la reconciliacion."""
    df = spark.read.parquet(str(carpeta_refinada))
    df = df.selectExpr(
        "date_format(timestamp_seconds(`timestamp` div 1000000000), 'yyyy-MM-dd') AS fecha",
        "sector",
        "consumo_m3",
    )
    return df.rdd.map(lambda r: ((r.fecha, r.sector), float(r.consumo_m3)))


In [27]:
# Autoverificacion de leer_refinada sobre la BASE (debe reproducir lo del Nivel 1)
spark = cs.crear_sesion("local[4]", DRIVER_MEMORY, "true")      # si la sesion estaba cerrada, se abre otra
pares_base = leer_refinada(spark, REFINADA)
clave_muestra, _ = pares_base.first()
assert pares_base.count() == 576_000, "leer_refinada debe devolver las 576 000 lecturas, sin filtro de calidad"
assert len(clave_muestra) == 2 and len(clave_muestra[0]) == 10 and clave_muestra[0][4] == "-" and clave_muestra[1].startswith("S-"), \
    "la clave debe ser (fecha AAAA-MM-DD, sector)"
print("leer_refinada correcta: 576 000 pares con clave", clave_muestra)

leer_refinada correcta: 576 000 pares con clave ('2024-01-11', 'S-NORTE')


**N2.2b · `agregar`**

In [28]:
def agregar(pares, metodo, particiones_mezcla=None):
    """Suma el consumo y cuenta las lecturas por (fecha, sector).

    Devuelve un RDD de tuplas (fecha, sector, consumo_total, n_lecturas).

    TODO (equipo): implementen las dos opciones de metodo.
      - "reduceByKey": combina dentro de cada particion antes de mezclar. Piensen que
        valor viaja por clave para poder sumar el consumo Y contar las lecturas.
      - "groupByKey": mezcla todos los valores y suma despues.
    Ambas reciben particiones_mezcla (None = por omision). Justifiquen en su registro de
    decisiones por que reduceByKey es valido aqui (funcion asociativa y conmutativa).

    Con reduceByKey viaja por clave el par (suma_consumo, n_lecturas): combinar dos
    parciales (a1, n1) y (a2, n2) en (a1 + a2, n1 + n2) es asociativo y conmutativo, asi
    que Spark puede combinar dentro de cada particion antes de mezclar (combinador del
    lado del mapa) y en cualquier orden despues. groupByKey, en cambio, mezcla cada
    consumo individual y suma al final."""
    if metodo == "reduceByKey":
        agregado = (pares.mapValues(lambda c: (c, 1))
                    .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]), numPartitions=particiones_mezcla))
        return agregado.map(lambda kv: (kv[0][0], kv[0][1], kv[1][0], kv[1][1]))
    if metodo == "groupByKey":
        agrupado = pares.groupByKey(numPartitions=particiones_mezcla)
        return agrupado.map(lambda kv: (kv[0][0], kv[0][1], float(sum(kv[1])), len(kv[1])))
    raise ValueError(f"metodo desconocido: {metodo}")


In [29]:
# Autoverificacion de agregar: los dos metodos deben reconciliar con la consolidada y respetar las particiones pedidas
referencia = cs.referencia_base(LAGO)
for metodo in ("reduceByKey", "groupByKey"):
    filas_metodo = agregar(pares_base, metodo).collect()
    ok_metodo, detalle_metodo = cs.reconciliar(filas_metodo, referencia)
    assert ok_metodo, f"{metodo} no reconcilia: {detalle_metodo}"
    assert len(filas_metodo) == 480 and {f[3] for f in filas_metodo} == {1200}, "480 grupos de 1 200 lecturas"
assert agregar(pares_base, "reduceByKey", 8).getNumPartitions() == 8, "particiones_mezcla fija las particiones despues de la mezcla"
print("agregar correcta: reduceByKey y groupByKey reconcilian con consolidada/ (480 grupos de 1 200 lecturas)")

agregar correcta: reduceByKey y groupByKey reconcilian con consolidada/ (480 grupos de 1 200 lecturas)


**N2.2c · `escribir_salida`**

In [30]:
def escribir_salida(spark, filas, carpeta_salida):
    """Escribe el resultado en carpeta_salida de forma que repetir la carga NO duplique filas.

    TODO (equipo):
      - Conviertan las filas a un DataFrame con nombres de columna fecha, sector,
        consumo_total_m3 y n_lecturas (fecha como DATE).
      - Decidan cuantos archivos escribir (480 filas) y con que modo de escritura; el
        modo debe hacer idempotente la carga. Expliquen la decision en su registro.

    Decision (equipo-1): un solo archivo (coalesce(1)) y mode("overwrite"). Son 480 filas
    (unos 10 KiB): partirlas por fecha dejaria 120 archivos diminutos y 120 carpetas que
    cuestan mas en metadatos que en datos. overwrite reemplaza la carpeta completa, asi
    que repetir la carga deja el mismo contenido; un append duplicaria las 480 filas en
    cada corrida.
    """
    from pyspark.sql.types import DateType, DoubleType, LongType, StringType, StructField, StructType
    esquema = StructType([
        StructField("fecha", DateType(), False),
        StructField("sector", StringType(), False),
        StructField("consumo_total_m3", DoubleType(), False),
        StructField("n_lecturas", LongType(), False),
    ])
    datos = [(dt.date.fromisoformat(f[0]), f[1], round(float(f[2]), 6), int(f[3]))
             for f in sorted(filas, key=lambda f: (f[0], f[1]))]
    (spark.createDataFrame(datos, schema=esquema)
     .coalesce(1)
     .write.mode("overwrite")
     .parquet(str(carpeta_salida)))


In [31]:
# Autoverificacion de escribir_salida: dos escrituras seguidas deben dejar 480 filas, sin duplicados y en un solo archivo
carpeta_prueba = Path(tempfile.mkdtemp()) / "salida_prueba"
filas_prueba = agregar(pares_base, "reduceByKey").collect()
for _ in range(2):
    escribir_salida(spark, filas_prueba, carpeta_prueba)
leido = pd.read_parquet(carpeta_prueba)
assert list(leido.columns) == ["fecha", "sector", "consumo_total_m3", "n_lecturas"], "las columnas deben llamarse fecha, sector, consumo_total_m3 y n_lecturas"
assert len(leido) == 480 and leido.duplicated(["fecha", "sector"]).sum() == 0, "repetir la escritura no debe duplicar filas"
assert len(list(carpeta_prueba.glob("part-*.parquet"))) == 1, "con 480 filas, un solo archivo"
assert cs.hash_salida([(str(r.fecha), r.sector, r.consumo_total_m3, int(r.n_lecturas)) for r in leido.itertuples()]) == cs.hash_salida(filas_prueba), \
    "el contenido escrito debe ser el mismo que el calculado"
print("escribir_salida correcta: dos escrituras, 480 filas, un archivo, sin duplicados")

escribir_salida correcta: dos escrituras, 480 filas, un archivo, sin duplicados


**N2.2d · Las tres funciones, en el script**

In [32]:
# Autoverificacion: lleven sus tres funciones a pipeline/carga_spark.py (reemplacen cada NotImplementedError) y recarguen el modulo
importlib.reload(cs)
for nombre in ("leer_refinada", "agregar", "escribir_salida"):
    assert "NotImplementedError" not in inspect.getsource(getattr(cs, nombre)), \
        f"{nombre} sigue sin completar en pipeline/carga_spark.py: copien aqui su version"
print("pipeline/carga_spark.py esta completo: las tres funciones ya no lanzan NotImplementedError")

pipeline/carga_spark.py esta completo: las tres funciones ya no lanzan NotImplementedError


### N2.3 Carga con el script: dos corridas por volumen

`correr_carga` ejecuta la función `main` del script, el equivalente de `python pipeline/carga_spark.py --equipo ... --volumen ... --metodo ... --particiones-mezcla ...`. Cada corrida anexa una línea a `bitacora_carga_spark.jsonl`. Dos corridas sobre el mismo volumen deben dejar **el mismo contenido** (mismo número de filas, mismas sumas, mismo hash); el nombre del archivo `part-*.parquet` cambia y no cuenta.

In [33]:
# N2.3 Ayudante: ejecutar la carga del script (su funcion main) y leer la linea que anexo a la bitacora
def correr_carga(volumen, metodo="reduceByKey", particiones_mezcla=None, aqe="true", ruta="parquet_refinada"):
    """Equivale a:  python pipeline/carga_spark.py --equipo EQUIPO --volumen V --metodo M [--particiones-mezcla N]
    main() cierra la sesion de Spark al terminar: la interfaz de esa corrida desaparece, pero sus cifras
    (etapas, tareas, bytes de mezcla) quedan en la bitacora."""
    argv = ["--equipo", EQUIPO, "--raiz", str(RAIZ), "--volumen", volumen, "--metodo", metodo, "--ruta", ruta,
            "--aqe", aqe, "--master", "local[4]", "--driver-memory", DRIVER_MEMORY]
    if particiones_mezcla is not None:
        argv += ["--particiones-mezcla", str(particiones_mezcla)]
    with contextlib.redirect_stdout(io.StringIO()):          # main imprime el registro: aqui se lee de la bitacora
        cs.main(argv)
    return json.loads((RAIZ / "bitacora_carga_spark.jsonl").read_text(encoding="utf-8").splitlines()[-1])


def diferencia_con_referencia(carpeta_salida, referencia):
    """Compara lo ESCRITO en disco con la referencia: filas, lecturas por sector y dia y diferencia maxima de la suma."""
    salida = pd.read_parquet(carpeta_salida).assign(fecha=lambda t: t["fecha"].astype(str))
    ref = pd.DataFrame(referencia, columns=["fecha", "sector", "consumo_ref", "lecturas_ref"])
    unido = salida.merge(ref, on=["fecha", "sector"], how="outer", indicator=True)
    return {"filas_salida": len(salida), "filas_referencia": len(ref), "solo_en_un_lado": int((unido["_merge"] != "both").sum()),
            "lecturas_min": int(salida["n_lecturas"].min()), "lecturas_max": int(salida["n_lecturas"].max()),
            "diferencia_max_consumo": float((unido["consumo_total_m3"] - unido["consumo_ref"]).abs().max())}

In [34]:
# N2.3 Dos corridas por volumen, SIN filtro de calidad: la salida debe tener el mismo contenido y reconciliar
referencias = {"base": cs.referencia_base(LAGO), "ampliacion": cs.referencia_ampliacion(EQUIPO_CARPETA)}
carpetas = {"base": LAGO / "consolidada_spark", "ampliacion": EQUIPO_CARPETA / "consolidada_spark"}
corridas, diferencias = [], {}
for volumen in ("base", "ampliacion"):
    for corrida in (1, 2):
        registro = correr_carga(volumen)
        salida = pd.read_parquet(carpetas[volumen])
        corridas.append({"volumen": volumen, "corrida": corrida, "filas": len(salida),
                         "consumo_total_m3": round(float(salida["consumo_total_m3"].sum()), 3),
                         "hash_salida": registro["hash_salida"][:16], "duplicados": int(salida.duplicated(["fecha", "sector"]).sum()),
                         "archivos_part": len(list(carpetas[volumen].glob("part-*.parquet"))),
                         "reconciliacion": registro["reconciliacion"], "tiempo_s": registro["tiempo_s"]})
    diferencias[volumen] = diferencia_con_referencia(carpetas[volumen], referencias[volumen])
tabla_idempotencia = pd.DataFrame(corridas)
print(tabla_idempotencia.to_string(index=False))
for volumen, d in diferencias.items():
    print(volumen, d)

   volumen  corrida  filas  consumo_total_m3      hash_salida  duplicados  archivos_part reconciliacion  tiempo_s
      base        1    480         92166.022 e3e31f0e1f783725           0              1             OK     1.336
      base        2    480         92166.022 e3e31f0e1f783725           0              1             OK     2.012
ampliacion        1    480        160402.481 a675b5e7fff0cd84           0              1             OK     2.689
ampliacion        2    480        160402.481 a675b5e7fff0cd84           0              1             OK     2.761
base {'filas_salida': 480, 'filas_referencia': 480, 'solo_en_un_lado': 0, 'lecturas_min': 1200, 'lecturas_max': 1200, 'diferencia_max_consumo': 2.842170943040401e-14}
ampliacion {'filas_salida': 480, 'filas_referencia': 480, 'solo_en_un_lado': 0, 'lecturas_min': 1824, 'lecturas_max': 2880, 'diferencia_max_consumo': 0.0}


In [35]:
# Autoverificacion: misma salida (mismo contenido) y reconciliacion sin filtro
for volumen in ("base", "ampliacion"):
    dos = tabla_idempotencia[tabla_idempotencia["volumen"] == volumen]
    assert dos["hash_salida"].nunique() == 1, f"las dos corridas de {volumen} deben dejar el mismo hash"
    assert (dos["filas"] == 480).all() and (dos["duplicados"] == 0).all() and (dos["archivos_part"] == 1).all()
    assert (dos["reconciliacion"] == "OK").all(), f"la carga de {volumen} debe reconciliar con su referencia"
    assert diferencias[volumen]["solo_en_un_lado"] == 0 and diferencias[volumen]["diferencia_max_consumo"] <= 1e-6
assert diferencias["base"]["lecturas_min"] == diferencias["base"]["lecturas_max"] == 1200, "la base tiene 1 200 lecturas por sector y dia"
print("Autoverificacion superada: dos corridas por volumen con el mismo hash, 480 filas, sin duplicados y reconciliadas con tolerancia 1e-6.")

Autoverificacion superada: dos corridas por volumen con el mismo hash, 480 filas, sin duplicados y reconciliadas con tolerancia 1e-6.


### N2.4 Barrido de particiones de mezcla

Antes de medir, escriban una hipótesis por cada configuración: qué esperan ver en tareas, bytes de mezcla y tiempo, y por qué. La celda del barrido no se ejecuta si las hipótesis siguen siendo `TODO`. Cada configuración corre tres veces y el tiempo se reporta como mediana: la variación entre corridas (el reloj varía hasta ×3 según la carga del equipo) puede ser mayor que la diferencia entre valores cercanos de $n$.

In [36]:
# N2.4 Escriban su hipotesis ANTES de ejecutar el barrido: que esperan ver (tareas, bytes, tiempo) y por que
# Escritas antes de medir, con lo que sabiamos de la ampliacion de equipo-1: 1 002 240 lecturas en 120 Parquet
# (unos 56 MiB), 480 claves (fecha, sector) y 4 particiones de entrada en el Nivel 1 de la base.
CONFIGURACIONES = [4, 8, 50, 200]      # valores de --particiones-mezcla: al menos tres distintos (pueden cambiarlos)
REPETICIONES = 3                       # corridas por configuracion: el tiempo se reporta como mediana
HIPOTESIS = {
    4: ("Esperamos 4 particiones de entrada + 4 de mezcla = 8 tareas en 2 etapas. Con reduceByKey cada particion de "
        "entrada combina antes de mezclar y emite a lo sumo un registro por clave (fecha, sector) que contiene, asi que "
        "los bytes de mezcla seran del orden de 10 a 20 KB (unos 480 registros), los mismos que en la base aunque la "
        "ampliacion tenga 1,7 veces mas lecturas. Es la configuracion mas rapida: 4 ranuras, 4 tareas por etapa, una ola."),
    8: ("12 tareas (4 + 8). Los bytes de mezcla no deberian cambiar respecto de 4: dependen de cuantas claves salen de "
        "cada particion de entrada, no de n. El tiempo sera casi igual: con local[4] las 8 tareas de reduccion corren "
        "en dos olas, pero cada una procesa ~60 claves y dura milisegundos."),
    50: ("54 tareas (4 + 50). Bytes de mezcla iguales o apenas mayores (cada bloque de mezcla tiene encabezado). "
         "Esperamos un tiempo algo mayor que con 4 u 8 por el costo fijo de planificar 50 tareas de ~10 claves cada una."),
    200: ("204 tareas (4 + 200): sobre-particionado, con ~2,4 claves por tarea y muchas tareas casi vacias. Bytes de "
          "mezcla similares. Esperamos el mayor tiempo del barrido (quizas +0,5 a 1 s) porque el costo de planificacion "
          "y de abrir 200 bloques de mezcla domina sobre el computo."),
}


In [37]:
# N2.4 Barrido de particiones de mezcla con reduceByKey sobre la ampliacion: REPETICIONES corridas por configuracion
assert set(HIPOTESIS) == set(CONFIGURACIONES), "escriban una hipotesis por cada configuracion del barrido"
assert all(h.strip() and not h.startswith("TODO") for h in HIPOTESIS.values()), "las hipotesis se escriben ANTES de medir: completen los TODO"
assert len(set(CONFIGURACIONES)) >= 3, "el barrido necesita al menos tres configuraciones distintas"

registros_barrido = [correr_carga("ampliacion", "reduceByKey", n) for n in CONFIGURACIONES for _ in range(REPETICIONES)]
barrido = pd.DataFrame(registros_barrido)
resumen_barrido = (barrido.groupby("particiones_mezcla")
                   .agg(particiones_entrada=("particiones_entrada", "first"), etapas=("etapas", "first"), tareas=("tareas", "first"),
                        mezcla_escritura_bytes=("mezcla_escritura_bytes", "first"), mezcla_lectura_bytes=("mezcla_lectura_bytes", "first"),
                        tiempo_mediana_s=("tiempo_s", "median"), tiempo_min_s=("tiempo_s", "min"), tiempo_max_s=("tiempo_s", "max"),
                        hashes_distintos=("hash_salida", "nunique"))
                   .reset_index())
print(resumen_barrido.to_string(index=False))

 particiones_mezcla  particiones_entrada  etapas  tareas  mezcla_escritura_bytes  mezcla_lectura_bytes  tiempo_mediana_s  tiempo_min_s  tiempo_max_s  hashes_distintos
                  4                    4       2       8                    9822                  9822             2.587         2.577         2.600                 1
                  8                    4       2      12                   11890                 11890             3.142         2.895         3.485                 1
                 50                    4       2      54                   26485                 26485             4.043         3.960         4.092                 1
                200                    4       2     204                   41318                 41318             7.241         6.827         7.281                 1


In [38]:
# Autoverificacion del barrido: bytes y tareas son estables; el contenido no depende de n
assert (resumen_barrido["hashes_distintos"] == 1).all() and barrido["hash_salida"].nunique() == 1, "el hash de la salida no debe depender de n"
assert (barrido["reconciliacion"] == "OK").all()
assert (resumen_barrido["tareas"] == resumen_barrido["particiones_entrada"] + resumen_barrido["particiones_mezcla"]).all(), \
    "con reduceByKey, tareas = particiones de entrada + n"
assert (resumen_barrido["etapas"] == 2).all()
print("Autoverificacion del barrido superada: tareas = entrada + n, dos etapas y el mismo hash en todas las configuraciones.")

Autoverificacion del barrido superada: tareas = entrada + n, dos etapas y el mismo hash en todas las configuraciones.


**Operación de agregación, caché y formato de entrada sobre la ampliación.** Las tres celdas siguientes dan las cifras propias que necesitan para decidir la operación de agregación (`reduceByKey` frente a `groupByKey`), el uso de caché y el formato de entrada (la corrida con `--ruta csv_contraste` lee la capa cruda, y no cuenta como configuración del barrido). La decisión es de su equipo.

In [39]:
# N2.4 (continuacion) El mismo trabajo con groupByKey sobre la ampliacion, para decidir la operacion de agregacion con cifras propias
registro_reduce = correr_carga("ampliacion", "reduceByKey")      # mezcla en las particiones de entrada (valor por omision)
registro_group = correr_carga("ampliacion", "groupByKey")
metodos = pd.DataFrame([{k: r[k] for k in ("metodo", "particiones_entrada", "particiones_mezcla", "etapas", "tareas", "mezcla_escritura_bytes", "tiempo_s", "hash_salida")}
                        for r in (registro_reduce, registro_group)])
razon_metodos = round(registro_group["mezcla_escritura_bytes"] / registro_reduce["mezcla_escritura_bytes"], 1)
print(metodos.to_string(index=False))
print(f"groupByKey escribe {razon_metodos} veces mas bytes de mezcla que reduceByKey sobre la ampliacion; mismo hash: {registro_group['hash_salida'] == registro_reduce['hash_salida']}")

     metodo  particiones_entrada  particiones_mezcla  etapas  tareas  mezcla_escritura_bytes  tiempo_s                                                      hash_salida
reduceByKey                    4                   4       2       8                    9822     3.282 a675b5e7fff0cd84fc0a3eabfea9d021fe04fa2375940c778f175c00f4ecc9f0
 groupByKey                    4                   4       2       8                 3240135     3.094 a675b5e7fff0cd84fc0a3eabfea9d021fe04fa2375940c778f175c00f4ecc9f0
groupByKey escribe 329.9 veces mas bytes de mezcla que reduceByKey sobre la ampliacion; mismo hash: True


In [40]:
# N2.4 (continuacion) Cache sobre la ampliacion: las mismas tres acciones del sub-paso E, con y sin cache
spark = cs.crear_sesion("local[4]", DRIVER_MEMORY, "true")
df_equipo = spark.read.parquet(str(EQUIPO_REFINADA))
ok_equipo = (df_equipo.filter("quality_flag = 'OK'")
                      .selectExpr("meter_id", "timestamp_seconds(`timestamp` DIV 1000000000) AS ts", "sector", "consumo_m3", "presion_bar")
                      .rdd.map(lambda f: (f.meter_id, f.ts, f.sector, f.consumo_m3, f.presion_bar)))
n_ok_equipo = ok_equipo.count()
t_sin_equipo = tres_acciones(ok_equipo)
ok_equipo.cache()
t_con_equipo = tres_acciones(ok_equipo)
almacen_equipo = api("storage/rdd")[0]
mib_equipo = round(almacen_equipo["memoryUsed"] / 2**20, 1)
ok_equipo.unpersist()
print(f"Lecturas OK de la ampliacion: {n_ok_equipo} de {perfil['lecturas_totales']}")
print("SIN cache (s):", t_sin_equipo, "· total", round(sum(t_sin_equipo), 1))
print("CON cache (s):", t_con_equipo, "· total", round(sum(t_con_equipo), 1))
print(f"Storage: {mib_equipo} MiB en {almacen_equipo['numCachedPartitions']} de {almacen_equipo['numPartitions']} particiones · disco {almacen_equipo['diskUsed']} B")

Lecturas OK de la ampliacion: 942031 de 1002240
SIN cache (s): [3.07, 3.16, 3.02] · total 9.2
CON cache (s): [3.89, 0.29, 0.3] · total 4.5
Storage: 16.4 MiB en 4 de 4 particiones · disco 0 B


In [41]:
# N2.4 (continuacion) Formato de entrada: la misma carga leyendo la capa CRUDA (CSV con textFile) frente a la refinada (Parquet)
registro_csv = correr_carga("ampliacion", "reduceByKey", ruta="csv_contraste")      # esta ruta NO cuenta como configuracion del barrido
formato = pd.DataFrame([{k: r[k] for k in ("ruta", "particiones_entrada", "etapas", "tareas", "mezcla_escritura_bytes", "tiempo_s", "reconciliacion", "hash_salida")}
                        for r in (registro_reduce, registro_csv)])
razon_tareas_formato = round(registro_csv["tareas"] / registro_reduce["tareas"], 1)
print(formato.to_string(index=False))
print(f"Leer el CSV lanza {registro_csv['tareas']} tareas frente a {registro_reduce['tareas']} con Parquet ({razon_tareas_formato} veces); mismo hash: {registro_csv['hash_salida'] == registro_reduce['hash_salida']}")

            ruta  particiones_entrada  etapas  tareas  mezcla_escritura_bytes  tiempo_s reconciliacion                                                      hash_salida
parquet_refinada                    4       2       8                    9822     3.282             OK a675b5e7fff0cd84fc0a3eabfea9d021fe04fa2375940c778f175c00f4ecc9f0
   csv_contraste                  120       2     240                   50501     8.251             OK a675b5e7fff0cd84fc0a3eabfea9d021fe04fa2375940c778f175c00f4ecc9f0
Leer el CSV lanza 240 tareas frente a 8 con Parquet (30.0 veces); mismo hash: True


### N2.5 La tarea que lee más bytes

Con `groupByKey`, clave **solo sector** y `numPartitions=8`, cada uno de los cuatro sectores queda en su propia tarea de la etapa de reducción. La señal fiable de la tarea "más cargada" es el volumen que lee (Shuffle Read Size), no su duración: el tiempo discrimina poco (entre el 4 % y el 24 % de diferencia) y se confunde con el ruido. El script de la carga agrega siempre por (fecha, sector), así que esta medición **no se hace por línea de comandos**: se hace solo en la celda `bytes_por_tarea`. La primera celda mide; la segunda es su turno.

In [42]:
# N2.5 Ayudante: cuanto leyo cada tarea de la etapa de reduccion cuando la clave es SOLO el sector
def bytes_por_tarea(metodo, particiones=8):
    """Agrega por sector (4 claves) con 'particiones' particiones de mezcla y lee de la interfaz, por tarea de la
    etapa de reduccion: bytes leidos (Shuffle Read Size), tiempo y sectores que recibio."""
    global spark
    spark = cs.crear_sesion("local[4]", DRIVER_MEMORY, "true")
    pares_sector = leer_refinada(spark, EQUIPO_REFINADA).map(lambda kv: (kv[0][1], kv[1]))      # (sector, consumo)
    if metodo == "groupByKey":
        resultado = pares_sector.groupByKey(numPartitions=particiones).map(lambda kv: (kv[0], sum(kv[1]), len(kv[1])))
    else:
        resultado = (pares_sector.map(lambda kv: (kv[0], (kv[1], 1)))
                                 .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]), numPartitions=particiones)
                                 .map(lambda kv: (kv[0], kv[1][0], kv[1][1])))
    etapa0 = cs.siguiente_etapa(spark)
    resultado.collect()
    time.sleep(0.5)
    reduccion = [e for e in api("stages") if e["stageId"] >= etapa0 and e["status"] == "COMPLETE" and e["shuffleReadBytes"] > 0][0]
    tareas = api(f"stages/{reduccion['stageId']}/{reduccion['attemptId']}/taskList?length=1000")
    sectores = dict(resultado.mapPartitionsWithIndex(lambda i, it: [(i, sorted(k for k, _, _ in it))]).collect())
    filas = []
    for t in sorted(tareas, key=lambda t: t["index"]):
        lectura = t["taskMetrics"]["shuffleReadMetrics"]
        filas.append({"tarea": t["index"], "bytes_leidos": lectura["remoteBytesRead"] + lectura["localBytesRead"],
                      "tiempo_ms": t["taskMetrics"]["executorRunTime"], "sectores": sectores[t["index"]]})
    return pd.DataFrame(filas)


tareas_group = bytes_por_tarea("groupByKey", 8)
tareas_reduce = bytes_por_tarea("reduceByKey", 8)
print("groupByKey, clave sector, 8 particiones de mezcla:")
print(tareas_group.to_string(index=False))
print("reduceByKey, misma clave y las mismas 8 particiones:")
print(tareas_reduce.to_string(index=False))

groupByKey, clave sector, 8 particiones de mezcla:
 tarea  bytes_leidos  tiempo_ms  sectores
     0             0        118        []
     1        705753        173 [S-NORTE]
     2        706246        288 [S-OESTE]
     3        706272        172  [S-ESTE]
     4             0        158        []
     5             0        158        []
     6             0        112        []
     7       1113066        139   [S-SUR]
reduceByKey, misma clave y las mismas 8 particiones:
 tarea  bytes_leidos  tiempo_ms  sectores
     0             0         80        []
     1           364         78 [S-NORTE]
     2           364         87 [S-OESTE]
     3           360         80  [S-ESTE]
     4             0         79        []
     5             0         74        []
     6             0         75        []
     7           364         79   [S-SUR]


In [43]:
# Su turno: identifiquen la tarea que LEE MAS BYTES (no la mas lenta) y calculen cuantas veces mas lee que la menor tarea con datos
def tarea_con_mas_bytes(tabla):
    """Devuelve un diccionario con la fila de la tabla cuya tarea leyo mas bytes y la razon entre esa tarea y
    la menor tarea que recibio datos. Las tareas con 0 bytes no recibieron datos y no cuentan para la razon."""
    con_datos = tabla[tabla["bytes_leidos"] > 0]
    fila = con_datos.loc[con_datos["bytes_leidos"].idxmax()]
    menor = int(con_datos["bytes_leidos"].min())
    return {"tarea": int(fila["tarea"]), "bytes_leidos": int(fila["bytes_leidos"]),
            "sectores": list(fila["sectores"]), "razon": round(int(fila["bytes_leidos"]) / menor, 2)}


mas_bytes = tarea_con_mas_bytes(tareas_group)
print(mas_bytes)


{'tarea': 7, 'bytes_leidos': 1113066, 'sectores': ['S-SUR'], 'razon': 1.58}


In [44]:
# Autoverificacion: la tarea que mas bytes lee recibe al sector dominante del perfil
dominante = perfil["sector_dominante"]
assert mas_bytes["sectores"] == [dominante], f"la tarea con mas bytes debe recibir solo al sector dominante ({dominante})"
assert mas_bytes["bytes_leidos"] == int(tareas_group["bytes_leidos"].max()) and mas_bytes["razon"] > 1
assert (tareas_group["bytes_leidos"] > 0).sum() == 4, "con 8 particiones, cada uno de los 4 sectores queda en su propia tarea"
assert tareas_reduce["bytes_leidos"].max() < 2_000, "con reduceByKey cada tarea lee cientos de bytes: el sesgo casi no se ve"
mas_lenta = int(tareas_group.loc[tareas_group["tiempo_ms"].idxmax(), "tarea"])
print(f"Autoverificacion superada: la tarea {mas_bytes['tarea']} lee {mas_bytes['bytes_leidos']} B ({mas_bytes['razon']} veces la menor con datos) y recibe a {dominante}.")
print("La tarea mas lenta es la", mas_lenta, "(coincide con la de mas bytes)" if mas_lenta == mas_bytes["tarea"] else "(NO coincide con la de mas bytes: por eso el tiempo no decide)")

Autoverificacion superada: la tarea 7 lee 1113066 B (1.58 veces la menor con datos) y recibe a S-SUR.
La tarea mas lenta es la 2 (NO coincide con la de mas bytes: por eso el tiempo no decide)


### N2.6 Contraste con el `groupBy` de DataFrame (solo si su barrido usó DataFrame)

Con la ejecución adaptativa activa, la etapa de reducción puede tener una sola tarea aunque `spark.sql.shuffle.partitions` valga 200; con `spark.sql.adaptive.enabled=false` se ejecutan tantas tareas como $n$. Si su equipo no usó DataFrame, salten esta celda y escriban "no se usó el `groupBy` de DataFrame" en la conclusión.

In [45]:
# N2.6 (solo si su barrido uso DataFrame) groupBy de DataFrame sobre la ampliacion, con la ejecucion adaptativa activa y desactivada
spark = cs.crear_sesion("local[4]", DRIVER_MEMORY, "true")
df_equipo = spark.read.parquet(str(EQUIPO_REFINADA))
contraste_df = pd.DataFrame([groupby_dataframe(df_equipo, "true", 200), groupby_dataframe(df_equipo, "false", 200),
                             groupby_dataframe(df_equipo, "false", 8)])
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "200")
print(contraste_df.to_string(index=False))
assert contraste_df.iloc[0]["tareas_reduccion"] == 1 and list(contraste_df["tareas_reduccion"][1:]) == [200, 8]
assert (contraste_df["filas"] == 480).all(), "el resultado es el mismo: 480 grupos"

  aqe  shuffle.partitions  filas  tareas_reduccion  mezcla_bytes
 true                 200    480                 1         38255
false                 200    480               200         38255
false                   8    480                 8         13373


### N2.7 Evidencia y registro de decisiones

Primero se escriben las conclusiones (lo **observado**, con la cifra y su pestaña, separado de lo **interpretado**, la causa). Después una celda genera `evidencia_nivel2_<equipo>.md` con el perfil, las tablas, las hipótesis y las conclusiones. El registro de decisiones usa el formato **ADR** (*architecture decision record*: un texto breve por decisión, con contexto, opciones, evidencia, decisión, alternativa descartada con su costo, consecuencias y la cifra que obligaría a revisarla, que es el formato de la práctica anterior) y justifica con cifras de **su ampliación** al menos tres de las cinco decisiones; una de ellas, el número de particiones.

**Antes de la celda de evidencia.** Esa celda usa variables de celdas anteriores (`tabla_idempotencia`, `resumen_barrido`, `metodos`, `formato`, `tareas_group`...). Si reinician el núcleo, esas variables se pierden: vuelvan a ejecutar las celdas que las producen, o guarden sus tablas antes de reiniciar.

In [46]:
# N2.7 Conclusiones del equipo: separen lo OBSERVADO (cifras de la tabla) de lo INTERPRETADO (la causa)
CONCLUSIONES = {
    4: ("OBSERVADO: 8 tareas (4 + 4) en 2 etapas, 9 822 B de mezcla escritos y leidos, tiempo mediano de 2,6 s (rango 2,58 a 2,60), el menor "
        "del barrido (pestana Stages). INTERPRETADO: se confirma la hipotesis de tareas y tiempo. Los 9 822 B son unos 480 "
        "registros combinados: el combinador de reduceByKey reduce 1 002 240 lecturas a un registro por clave y por "
        "particion de entrada. Una sola ola de 4 tareas por etapa aprovecha exactamente las 4 ranuras."),
    8: ("OBSERVADO: 12 tareas, 11 890 B de mezcla (+21 % frente a n = 4) y un tiempo mediano de 3,1 s, +0,5 s frente a n = 4 "
        "(dentro de la variacion del reloj). INTERPRETADO: la hipotesis de tiempo se confirma a grandes rasgos, pero la de bytes 'iguales' se REFUTA en parte: "
        "los bytes crecen con n aunque los registros sean los mismos, porque cada tarea de mapa escribe un bloque por "
        "particion de mezcla (4 x 8 = 32 bloques), y cada bloque tiene su propio encabezado de serializacion y compresion."),
    50: ("OBSERVADO: 54 tareas, 26 485 B de mezcla (2,7 veces los de n = 4) y un tiempo mediano de 4,0 s (+56 %). "
         "INTERPRETADO: se confirma el costo de planificacion: 50 tareas de unas 10 claves cada una duran menos que su "
         "propio arranque. Los bytes crecen por los 4 x 50 = 200 bloques de mezcla; nuestra hipotesis de 'bytes iguales' "
         "subestimo ese costo fijo por bloque."),
    200: ("OBSERVADO: 204 tareas, 41 318 B de mezcla (4,2 veces los de n = 4) y un tiempo mediano de unos 7,2 s, mas del doble "
          "que n = 4. Es el mayor del barrido, con un rango de min. y max. que no se solapa con los de n = 4 u 8. "
          "INTERPRETADO: se confirma el sobre-particionado: unas 2,4 claves por tarea y 800 bloques de mezcla. El tiempo crecio "
          "mas de lo que estimamos (+0,5 a 1 s), asi que el costo por tarea en local[4] es de unos 25 ms."),
}
CONCLUSION_TAREA = ("OBSERVADO: con groupByKey, clave sector y 8 particiones, la tarea 7 (S-SUR) lee 1 113 066 B y las de S-NORTE, "
                    "S-OESTE y S-ESTE unos 706 000 B cada una: razon 1,58. Cuatro tareas no reciben datos. La tarea mas lenta fue "
                    "la 2 (S-OESTE, 288 ms), no la 7 (139 ms). Con reduceByKey cada tarea lee unos 364 B. INTERPRETADO: S-SUR es "
                    "el sector dominante de nuestra ampliacion, con 120 de 348 medidores (p = 0,345) frente a 76 en cada uno de "
                    "los otros tres. 120 / 76 = 1,58: los bytes leidos son proporcionales a las lecturas que mezcla groupByKey. "
                    "El tiempo no lo refleja porque las tareas duran ~100 ms y el ruido de planificacion domina, por eso la senal "
                    "fiable del sesgo son los bytes. Con reduceByKey el sesgo desaparece: cada sector llega ya combinado.")
CONCLUSION_DF = ("OBSERVADO: groupBy de DataFrame con AQE activa y shuffle.partitions = 200 ejecuto 1 sola tarea de reduccion "
                 "(38 255 B de mezcla); con AQE desactivada ejecuto 200; con 8 ejecuto 8 (13 373 B). INTERPRETADO: lo "
                 "configurado (200) no es lo ejecutado: AQE fusiona las particiones de mezcla pequenas despues de medirlas. "
                 "Nuestro barrido uso la API de RDD (reduceByKey), donde lo configurado si es lo ejecutado.")


In [47]:
# N2.7 Evidencia del Nivel 2: se genera el archivo evidencia_nivel2_<equipo>.md con lo medido y lo concluido
# Usa variables de celdas anteriores (tabla_idempotencia, resumen_barrido, metodos, formato, tareas_group, tareas_reduce, contraste_df...).
# Si reiniciaron el nucleo, esas variables ya no existen: vuelvan a ejecutar las celdas que las producen antes de esta.
assert set(CONCLUSIONES) == set(CONFIGURACIONES), "escriban una conclusion por cada configuracion del barrido"
assert all(c.strip() and not c.startswith("TODO") for c in CONCLUSIONES.values()) and not CONCLUSION_TAREA.startswith("TODO") and not CONCLUSION_DF.startswith("TODO"), \
    "completen las conclusiones: separen lo observado de lo interpretado"


def md_tabla(tabla):
    """Tabla en markdown, sin dependencias adicionales."""
    filas = ["| " + " | ".join(str(v) for v in fila) + " |" for fila in tabla.itertuples(index=False)]
    return "\n".join(["| " + " | ".join(map(str, tabla.columns)) + " |", "|" + "---|" * len(tabla.columns), *filas])


partes = [f"# Evidencia del Nivel 2 · {EQUIPO}", "",
          "## Perfil de la ampliacion", "",
          f"- Comando: `python practica/generar_ampliacion_s15.py --equipo {EQUIPO}` ({tiempo_generacion} s)",
          f"- Medidores: {perfil['medidores']} · lecturas (total, no incremento): {perfil['lecturas_totales']}",
          f"- Sector dominante: {perfil['sector_dominante']} · fraccion de medidores: {perfil['fraccion_dominante']} · disco: {disco_mib} MiB", "",
          "## Carga reejecutable y reconciliacion", "", md_tabla(tabla_idempotencia), "",
          "## Barrido de particiones de mezcla (reduceByKey, ampliacion)", "", md_tabla(resumen_barrido), ""]
for n in CONFIGURACIONES:
    fila = resumen_barrido[resumen_barrido["particiones_mezcla"] == n].iloc[0]
    partes += [f"### Configuracion: {n} particiones de mezcla", "",
               f"- **Hipotesis (escrita antes de medir):** {HIPOTESIS[n]}",
               f"- **Configuracion exacta:** `python pipeline/carga_spark.py --equipo {EQUIPO} --volumen ampliacion --metodo reduceByKey --particiones-mezcla {n}`",
               f"- **Medicion en la interfaz (pestana Stages):** {int(fila['tareas'])} tareas, Shuffle Write {int(fila['mezcla_escritura_bytes'])} B, "
               f"Shuffle Read {int(fila['mezcla_lectura_bytes'])} B; tiempo {fila['tiempo_mediana_s']:.1f} s (mediana de {REPETICIONES})",
               f"- **Conclusion:** {CONCLUSIONES[n]}", ""]
partes += ["## Operacion de agregacion sobre la ampliacion", "", md_tabla(metodos), "",
           "## Formato de entrada: Parquet de la refinada frente a CSV de la cruda (ampliacion)", "", md_tabla(formato), "",
           "## Tarea que lee mas bytes (groupByKey, clave sector, 8 particiones de mezcla)", "", md_tabla(tareas_group), "",
           f"Con reduceByKey y la misma clave:", "", md_tabla(tareas_reduce), "",
           f"- **Conclusion:** {CONCLUSION_TAREA}", "",
           "## Contraste con el groupBy de DataFrame", ""]
if "contraste_df" in globals():
    partes += [md_tabla(contraste_df), ""]
partes += [f"- **Conclusion:** {CONCLUSION_DF}", ""]
ruta_evidencia = RAIZ / f"evidencia_nivel2_{EQUIPO}.md"
ruta_evidencia.write_text("\n".join(partes), encoding="utf-8")
print("Escrito:", ruta_evidencia, f"({ruta_evidencia.stat().st_size} bytes)")

Escrito: /proyecto/evidencia_nivel2_equipo-1.md (8749 bytes)


In [48]:
# Plantilla del registro de decisiones (ADR: un texto breve por decision). Copien este texto a su repositorio, una vez por decision, y completenlo con SUS cifras.
PLANTILLA_ADR = """# ADR-N: <titulo corto de la decision>
Estado: propuesta | aceptada
Contexto: <que se decide, con que volumen (cifras de su ampliacion) y que pregunta lo detono>
Opciones: <los valores o alternativas que midieron>
Evidencia (pestana y cifra de la interfaz): <tareas, bytes de mezcla, bytes por tarea, tiempo, Storage...>
Decision: <que eligieron y en que condiciones vale>
Alternativa descartada y su costo: <que no eligieron y que costaba, con cifra medida o calculada>
Consecuencias: <que se gana y que se arriesga con esta decision>
Cifra que obligaria a revisarla: <por ejemplo, un volumen o una proporcion de sesgo>"""
print(PLANTILLA_ADR)

# ADR-N: <titulo corto de la decision>
Estado: propuesta | aceptada
Contexto: <que se decide, con que volumen (cifras de su ampliacion) y que pregunta lo detono>
Opciones: <los valores o alternativas que midieron>
Evidencia (pestana y cifra de la interfaz): <tareas, bytes de mezcla, bytes por tarea, tiempo, Storage...>
Decision: <que eligieron y en que condiciones vale>
Alternativa descartada y su costo: <que no eligieron y que costaba, con cifra medida o calculada>
Consecuencias: <que se gana y que se arriesga con esta decision>
Cifra que obligaria a revisarla: <por ejemplo, un volumen o una proporcion de sesgo>


### N2.8 Qué se versiona

El repositorio del equipo lleva texto y resultados pequeños, no el paquete `.zip` ni las capas pesadas.

In [49]:
# Que se versiona en el repositorio del equipo: texto y resultados pequenos; NO el .zip ni las capas cruda/ y refinada/
for ruta in sorted(list(RAIZ.glob("datalake*/consolidada_spark")) + [RAIZ / "bitacora_carga_spark.jsonl", RAIZ / "mediciones_nivel1.csv",
                                                                   RAIZ / f"evidencia_nivel2_{EQUIPO}.md"]):
    tam = sum(f.stat().st_size for f in ruta.rglob("*") if f.is_file()) if ruta.is_dir() else ruta.stat().st_size
    print("SI se versiona :", ruta.relative_to(RAIZ), f"({tam / 1024:.1f} KiB)")
for ruta in (RAIZ / "datalake" / "cruda", RAIZ / "datalake" / "refinada", EQUIPO_CARPETA / "cruda", EQUIPO_CARPETA / "refinada"):
    tam = sum(f.stat().st_size for f in ruta.rglob("*") if f.is_file())
    print("NO se versiona :", ruta.relative_to(RAIZ), f"({tam / 2**20:.1f} MiB: se regenera con el generador o viene en el paquete)")

SI se versiona : bitacora_carga_spark.jsonl (11.1 KiB)
SI se versiona : datalake/consolidada_spark (4.8 KiB)
SI se versiona : datalake_equipo/consolidada_spark (4.8 KiB)
SI se versiona : evidencia_nivel2_equipo-1.md (8.5 KiB)
SI se versiona : mediciones_nivel1.csv (0.9 KiB)
NO se versiona : datalake/cruda (28.2 MiB: se regenera con el generador o viene en el paquete)
NO se versiona : datalake/refinada (2.8 MiB: se regenera con el generador o viene en el paquete)
NO se versiona : datalake_equipo/cruda (51.9 MiB: se regenera con el generador o viene en el paquete)
NO se versiona : datalake_equipo/refinada (4.2 MiB: se regenera con el generador o viene en el paquete)


## Nivel 3 · Frontera · opcional

**La decisión (sin guía).** Diseñar la carga **nocturna** de la capa cruda de su ampliación (120 CSV pequeños) dentro del límite de 3 GiB del contenedor: formato de entrada, número de particiones, caché, presupuesto de memoria y **una alternativa descartada con su costo**. La corrida `csv_contraste` del Nivel 2 ya dio la primera cifra del formato de entrada: la primera celda la amplía con tres formas más de leer la cruda; la segunda es el cálculo del presupuesto, que después contrastan con el pico que mide `docker stats` (la diferencia entre el cálculo y el pico es memoria que hay que explicar). Cada decisión se documenta con cifras en su registro de decisiones; contraste el presupuesto con `docker stats` en la terminal del computador.

In [50]:
# Nivel 3 (herramienta de medicion): tres formas de leer los 120 CSV de la capa cruda de su ampliacion
spark = cs.crear_sesion("local[4]", DRIVER_MEMORY, "true")
CRUDA_EQUIPO = EQUIPO_CARPETA / "cruda" / "acueducto"
patron_csv = str(CRUDA_EQUIPO / "*" / "*" / "*" / "lecturas.csv")


def medir_lectura(nombre, leer):
    """Cuenta particiones y lo que lanzan la lectura y el count() para una forma de leer la cruda."""
    trabajo = siguiente_trabajo()
    t0 = time.time()
    datos = leer()
    al_leer = actividad(trabajo)
    particiones = datos.rdd.getNumPartitions() if hasattr(datos, "rdd") else datos.getNumPartitions()
    filas = datos.count()
    total = actividad(trabajo)
    return {"lectura": nombre, "particiones": particiones, "trabajos_al_leer": al_leer["trabajos"], "trabajos_total": total["trabajos"],
            "tareas_total": total["tareas"], "filas": filas, "tiempo_s": round(time.time() - t0, 1)}


lecturas_cruda_n3 = pd.DataFrame([
    medir_lectura("textFile (texto, incluye 120 encabezados)", lambda: spark.sparkContext.textFile(patron_csv)),
    medir_lectura("DataFrame CSV sin inferir (todo texto)", lambda: spark.read.option("header", True).csv(str(CRUDA_EQUIPO))),
    medir_lectura("DataFrame CSV con inferSchema", lambda: spark.read.option("header", True).option("inferSchema", True).csv(str(CRUDA_EQUIPO))),
    medir_lectura("Parquet de la refinada", lambda: spark.read.parquet(str(EQUIPO_REFINADA))),
])
print(lecturas_cruda_n3.to_string(index=False))

                                  lectura  particiones  trabajos_al_leer  trabajos_total  tareas_total   filas  tiempo_s
textFile (texto, incluye 120 encabezados)          120                 0               1           120 1002360       4.5
   DataFrame CSV sin inferir (todo texto)            5                 2               4           127 1002240       2.4
            DataFrame CSV con inferSchema            5                 3               5           132 1002240       3.1
                   Parquet de la refinada            4                 1               3             6 1002240       1.3


In [51]:
# Nivel 3: presupuesto de memoria del contenedor (cifras en MiB). Es un calculo, no una medicion: despues lo contrastan con docker stats
def presupuesto_memoria(limite_mib, montones_mib, procesos_python, mib_por_proceso_python, jvm_fuera_del_monton_mib, kernel_mib):
    """Devuelve {"total_mib": ..., "margen_mib": ...}: la suma de los componentes y lo que sobra bajo el limite del contenedor.

    TODO (equipo): sumen el monton de la JVM, la memoria de la JVM fuera del monton, los procesos de Python
    (uno por ranura de local[N]) y el nucleo de Jupyter; el margen es el limite menos ese total.
    """
    total = montones_mib + jvm_fuera_del_monton_mib + procesos_python * mib_por_proceso_python + kernel_mib
    return {"total_mib": total, "margen_mib": limite_mib - total}


presupuesto = presupuesto_memoria(limite_mib=3072, montones_mib=1024, procesos_python=4, mib_por_proceso_python=90,
                                  jvm_fuera_del_monton_mib=300, kernel_mib=133)
print(presupuesto)

{'total_mib': 1817, 'margen_mib': 1255}


In [52]:
# Autoverificacion del Nivel 3 (la decision de diseno la valora quien revise; esto solo comprueba el calculo)
suma_esperada = 1024 + 300 + 4 * 90 + 133
assert presupuesto["total_mib"] == suma_esperada, "el total es el monton + la JVM fuera del monton + 4 procesos de Python + el nucleo de Jupyter"
assert presupuesto["margen_mib"] == 3072 - suma_esperada, "el margen es el limite del contenedor menos el total"
assert presupuesto_memoria(3072, 2048, 4, 90, 300, 133)["margen_mib"] < presupuesto["margen_mib"], "con un monton mayor el margen se reduce"
lectura_textfile = lecturas_cruda_n3.iloc[0]
assert lectura_textfile["particiones"] == 120, "con textFile hay una particion por archivo: 120"
print("Autoverificacion del Nivel 3 superada: el presupuesto cuadra y textFile da 120 particiones.")
print(f"Su presupuesto da {presupuesto['total_mib']} MiB y deja un margen de {presupuesto['margen_mib']} MiB bajo los 3 072 MiB del contenedor.")
print(f"Pero el cuaderno completo llega a 2 200 a 2 330 MiB en docker stats: el calculo se queda corto por {2200 - presupuesto['total_mib']} a {2330 - presupuesto['total_mib']} MiB.")
print("Esa brecha es memoria que el presupuesto no conto: expliquen de donde sale, corrijan el presupuesto y declaren el margen real.")

Autoverificacion del Nivel 3 superada: el presupuesto cuadra y textFile da 120 particiones.
Su presupuesto da 1817 MiB y deja un margen de 1255 MiB bajo los 3 072 MiB del contenedor.
Pero el cuaderno completo llega a 2 200 a 2 330 MiB en docker stats: el calculo se queda corto por 383 a 513 MiB.
Esa brecha es memoria que el presupuesto no conto: expliquen de donde sale, corrijan el presupuesto y declaren el margen real.


## Reto de diseño · 30 minutos · opcional

**Situación.** La gerencia proyecta llegar a 7,8 GB de lecturas por cierre. En una hoja (cálculo propio, sin exigir un asistente de IA) dimensionen: particiones $P=\lceil S/B\rceil$ con bloques de 128 MB, ejecutores $E$ y núcleos $c$, olas $W=\lceil P/(E\cdot c)\rceil$, memoria por tarea $\approx M/c$ y una postura razonada sobre clúster o máquina única, con el umbral de escala que vieron en el primer módulo. Declaren como supuesto el factor de expansión de un bloque en memoria.

In [53]:
# Reto de diseno (papel): dimensionar el procesamiento de 7,8 GB. Completen las tres funciones y declaren sus supuestos
import math


def particiones_necesarias(tamano_gb, bloque_mb):
    """P = techo(S / B): particiones para S gigabytes con bloques de B megabytes (1 GB = 1024 MB).

    TODO (equipo): devuelvan un entero.
    """
    return int(math.ceil(tamano_gb * 1024 / bloque_mb))


def olas(particiones, ejecutores, nucleos):
    """W = techo(P / (E * c)): rondas de tareas que hacen falta con E ejecutores de c nucleos.

    TODO (equipo): devuelvan un entero.
    """
    return int(math.ceil(particiones / (ejecutores * nucleos)))


def memoria_por_tarea_mib(monton_mib, nucleos, fraccion=0.6):
    """M / c con M = (H - 300 MiB) * fraccion: memoria de ejecucion y almacenamiento de que dispone cada tarea.

    TODO (equipo): devuelvan un decimal redondeado a una cifra.
    """
    return round((monton_mib - 300) * fraccion / nucleos, 1)


P = particiones_necesarias(7.8, 128)
W = olas(P, ejecutores=2, nucleos=4)
memoria_tarea = memoria_por_tarea_mib(4096, 4)
print(f"P = {P} particiones · W = {W} olas · {memoria_tarea} MiB por tarea")

P = 63 particiones · W = 8 olas · 569.4 MiB por tarea


In [54]:
# Autoverificacion del reto: se comprueban las propiedades de cada cifra (las formulas estan en el enunciado); los supuestos y la postura los valora quien revise
assert isinstance(P, int) and P * 128 >= 7.8 * 1024 > (P - 1) * 128, "P es el menor entero con P * B >= S (1 GB = 1024 MB)"
assert isinstance(W, int) and (W - 1) * (2 * 4) < P <= W * (2 * 4), "W es el menor entero con W * (E * c) >= P"
assert abs(memoria_tarea * 4 / 0.6 + 300 - 4096) < 0.1, "la memoria por tarea es (H - 300) * 0,6 / c, con una cifra decimal"
assert particiones_necesarias(1, 128) == 8 and olas(9, 1, 4) == 3 and memoria_por_tarea_mib(2048, 2) == 524.4, "las funciones tambien deben servir para otros valores"
print("Autoverificacion del reto superada. Declaren como supuesto el factor de expansion de un bloque en memoria y defiendan su postura con estas cifras.")

Autoverificacion del reto superada. Declaren como supuesto el factor de expansion de un bloque en memoria y defiendan su postura con estas cifras.


## Antes de entregar

Suban al repositorio del equipo: `pipeline/carga_spark.py` completo, `datalake/consolidada_spark/`, `datalake_equipo/consolidada_spark/`, `bitacora_carga_spark.jsonl`, `mediciones_nivel1.csv`, `evidencia_nivel2_<equipo>.md`, el registro de decisiones, el contraste del sub-paso A y el README con el identificador de equipo y los pasos para regenerar `datalake_equipo/`. La celda siguiente comprueba lo que se puede comprobar con código.

In [55]:
# Lista final de autoverificacion de la tarea: cada linea comprueba un entregable de los Niveles 1 y 2
def _g(nombre):
    return globals().get(nombre)


bitacora = pd.read_json(RAIZ / "bitacora_carga_spark.jsonl", lines=True)
barrido_amp = bitacora[(bitacora["volumen"] == "ampliacion") & (bitacora["ruta"] == "parquet_refinada") & (bitacora["metodo"] == "reduceByKey")]
comprobaciones = {
    "Nivel 1: sesion local[4] con 1g y 434,4 MiB de almacenamiento": _g("maxima_mib") == 434.4,
    "Nivel 1: error de nanosegundos provocado y resuelto": "Illegal Parquet type" in (_g("mensaje_nanos") or ""),
    "Nivel 1: bytes de mezcla de reduceByKey y groupByKey medidos": _g("razon_bytes") is not None and _g("razon_bytes") > 50,
    "Nivel 1: reconciliacion sin filtro contra consolidada/": bool(_g("ok_reduce")) and _g("lecturas_por_grupo") == [1200],
    "Nivel 1: tres acciones con y sin cache": _g("tiempos_con_cache") is not None and sum(_g("tiempos_con_cache")) < sum(_g("tiempos_sin_cache")),
    "Nivel 1: tabla de mediciones guardada": (RAIZ / "mediciones_nivel1.csv").exists(),
    "Nivel 2: ampliacion generada y datalake/ intacto": (EQUIPO_CARPETA / "totales_control.csv").exists() and hash_arbol(LAGO)[0] == _g("datalake_antes"),
    "Nivel 2: las tres funciones estan en pipeline/carga_spark.py": all("NotImplementedError" not in inspect.getsource(getattr(cs, n)) for n in ("leer_refinada", "agregar", "escribir_salida")),
    "Nivel 2: dos corridas por volumen con el mismo hash y reconciliadas": _g("tabla_idempotencia") is not None and all(
        g["hash_salida"].nunique() == 1 and (g["reconciliacion"] == "OK").all() for _, g in tabla_idempotencia.groupby("volumen")),
    "Nivel 2: salidas en datalake/ y datalake_equipo/ (consolidada_spark)": (LAGO / "consolidada_spark").is_dir() and (EQUIPO_CARPETA / "consolidada_spark").is_dir(),
    "Nivel 2: barrido de al menos tres configuraciones en la bitacora": barrido_amp["particiones_mezcla"].nunique() >= 3,
    "Nivel 2: corrida de contraste con la capa cruda (formato de entrada)": bool((bitacora["ruta"] == "csv_contraste").any()),
    "Nivel 2: tarea que lee mas bytes identificada": _g("mas_bytes") is not None and _g("mas_bytes")["sectores"] == [perfil["sector_dominante"]],
    "Nivel 2: evidencia_nivel2 generada con hipotesis y conclusion por configuracion": (RAIZ / f"evidencia_nivel2_{EQUIPO}.md").exists() and
        (RAIZ / f"evidencia_nivel2_{EQUIPO}.md").read_text(encoding="utf-8").count("**Hipotesis") >= 3,
}
for nombre, ok in comprobaciones.items():
    print("OK    " if ok else "FALTA ", nombre)
assert all(comprobaciones.values()), "Hay comprobaciones pendientes: revisen las lineas marcadas como FALTA"
print("\nTodas las comprobaciones automaticas pasaron. Falta lo que ningun codigo puede verificar: el registro de decisiones, el contraste del sub-paso A,")
print("la lectura en ingles y el README del repositorio con el identificador de equipo. El Nivel 3 y el reto son opcionales.")

OK     Nivel 1: sesion local[4] con 1g y 434,4 MiB de almacenamiento
OK     Nivel 1: error de nanosegundos provocado y resuelto
OK     Nivel 1: bytes de mezcla de reduceByKey y groupByKey medidos
OK     Nivel 1: reconciliacion sin filtro contra consolidada/
OK     Nivel 1: tres acciones con y sin cache
OK     Nivel 1: tabla de mediciones guardada
OK     Nivel 2: ampliacion generada y datalake/ intacto
OK     Nivel 2: las tres funciones estan en pipeline/carga_spark.py
OK     Nivel 2: dos corridas por volumen con el mismo hash y reconciliadas
OK     Nivel 2: salidas en datalake/ y datalake_equipo/ (consolidada_spark)
OK     Nivel 2: barrido de al menos tres configuraciones en la bitacora
OK     Nivel 2: corrida de contraste con la capa cruda (formato de entrada)
OK     Nivel 2: tarea que lee mas bytes identificada
OK     Nivel 2: evidencia_nivel2 generada con hipotesis y conclusion por configuracion

Todas las comprobaciones automaticas pasaron. Falta lo que ningun codigo puede verifica

In [56]:
# Cierre: se libera la memoria de la sesion de Spark. Despues apaguen el perfil desde la terminal del computador:
#   docker compose --profile s15 down
spark.stop()
print("Sesion de Spark cerrada.")

Sesion de Spark cerrada.
